# Repository setup

Run this setup cell first. Paths resolve from the repository root; see `docs/research.md` for data, model and environment configuration. Stored plots are preserved under `reports/figures/`.


In [ ]:
from pathlib import Path
import sys

# Locate the checkout when Jupyter starts from the root or notebooks/.
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "scripts" / "project_paths.py").is_file():
        scripts_dir = str(candidate / "scripts")
        if scripts_dir not in sys.path:
            sys.path.insert(0, scripts_dir)
        break
else:
    raise RuntimeError("Start Jupyter from this repository or its notebooks directory.")

from project_paths import (
    PROJECT_ROOT, RAW_DATA_DIR, SAMPLE_DIR, PRETRAINED_PATH,
    CHECKPOINT_ROOT, REPORT_DIR, configured_path, get_project_root,
)


# Arsitektur

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, re
import monai.networks.nets.resnet as monai_resnet
from monai.networks.layers.factories import Conv
from monai.networks.nets.basic_unet import TwoConv, UpCat

def prepare_medicalnet_state_for_monai(sd_in, in_channels=4):
    sd_raw = sd_in.get("state_dict", sd_in)
    sd = dict(sd_raw)
    remap = {}

    for k, v in sd.items():
        kk = k[len("module."):] if k.startswith("module.") else k

        # drop classifier
        if kk.startswith("fc."):
            continue

        # strip inner ".conv" (varian yang simpan conv submodule)
        kk = kk.replace(".conv1.conv.", ".conv1.")
        kk = kk.replace(".conv2.conv.", ".conv2.")
        kk = kk.replace(".conv3.conv.", ".conv3.")  # aman utk R50 juga

        # map style 'downsample.conv/bn' → 'downsample.0/1'
        kk = kk.replace(".downsample.conv.", ".downsample.0.")
        kk = kk.replace(".downsample.bn.",  ".downsample.1.")

        # map style 'shortcut' → 'downsample'
        kk = kk.replace(".shortcut.0.", ".downsample.0.")
        kk = kk.replace(".shortcut.1.", ".downsample.1.")

        # sebagian ckpt pakai 'downsample.weight' (tanpa .0) → asumsi conv
        if ".downsample.weight" in kk and ".downsample.0.weight" not in kk:
            kk = kk.replace(".downsample.weight", ".downsample.0.weight")
        if ".downsample.bias" in kk and ".downsample.0.bias" not in kk:
            kk = kk.replace(".downsample.bias", ".downsample.0.bias")

        remap[kk] = v

    # adapt conv1 Cin=1 → multi-modal (mis. 4)
    for conv1_key in ["conv1.weight", "conv1.conv.weight"]:
        if conv1_key in remap:
            w = remap[conv1_key]
            if w.ndim == 5 and w.shape[1] == 1 and in_channels > 1:
                w = w.mean(1, keepdim=True).repeat(1, in_channels, 1, 1, 1) / float(in_channels)
                remap[conv1_key] = w
                break

    return remap


@torch.no_grad()
def load_medicalnet_into_monai_resnet(backbone, pth_path, in_channels=4, verbose=True):
    raw = torch.load(pth_path, map_location="cpu")
    sd  = prepare_medicalnet_state_for_monai(raw, in_channels=in_channels)
    info = backbone.load_state_dict(sd, strict=False)
    try:
        missing, unexpected = info.missing_keys, info.unexpected_keys
    except AttributeError:
        missing, unexpected = info
    if verbose:
        monai_keys = backbone.state_dict()
        matched = sum(int(k in monai_keys and monai_keys[k].shape == sd[k].shape) for k in sd.keys())
        print(f"[MedicalNet→MONAI] matched={matched}  missing={len(missing)}  unexpected={len(unexpected)}")
    return info


class MedicalNetResNet18Encoder(nn.Module):
    def __init__(self, in_channels=4):
        super().__init__()
        self.backbone = monai_resnet.resnet18(
            spatial_dims=3, n_input_channels=in_channels, pretrained=False, progress=False
        )
        self.layer1 = self.backbone.layer1
        self.layer2 = self.backbone.layer2
        self.layer3 = self.backbone.layer3
        self.layer4 = self.backbone.layer4

    @torch.no_grad()
    def load_medicalnet_pretrain(self, pth_path, in_channels=4):
        load_medicalnet_into_monai_resnet(
            backbone=self.backbone, pth_path=pth_path, in_channels=in_channels, verbose=True
        )

    def forward(self, x):
        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = F.relu(x, inplace=True)
        s0 = x                 # <<< /2 tap (SEBELUM maxpool)
        x  = self.backbone.maxpool(x)
        s1 = self.layer1(x)    # /4
        s2 = self.layer2(s1)   # /8
        s3 = self.layer3(s2)   # /16
        return s0, s1, s2, s3


class UNetPP3D_Decoder(nn.Module):
    def __init__(
        self,
        spatial_dims=3,
        in_channels=4,
        out_channels=3,
        features=(16,32,64,128,128,16),
        act=("LeakyReLU", {"negative_slope": 0.1, "inplace": True}),
        norm=("instance", {"affine": True}),
        dropout=0.0,
        upsample="nontrainable",
        enc_channels=(64,128,256,512),
        deep_supervision=True,
    ):
        super().__init__()
        self.deep_supervision = deep_supervision
        F0,F1,F2,F3,F4,Fhead = features
        Conv1x1 = lambda a,b: Conv["conv", spatial_dims](a, b, kernel_size=1, bias=False)

        self.conv_0_0 = TwoConv(spatial_dims, in_channels, F0, act, norm, True, dropout)

        self.enc_adapt1 = Conv1x1(enc_channels[0], F1)
        self.enc_adapt2 = Conv1x1(enc_channels[1], F2)
        self.enc_adapt3 = Conv1x1(enc_channels[2], F3)
        self.enc_adapt4 = Conv1x1(enc_channels[3], F4)

        self.upcat_0_1 = UpCat(spatial_dims, F1, F0, F0, act, norm, True, dropout, upsample, halves=False)
        self.upcat_1_1 = UpCat(spatial_dims, F2, F1, F1, act, norm, True, dropout, upsample)
        self.upcat_2_1 = UpCat(spatial_dims, F3, F2, F2, act, norm, True, dropout, upsample)
        self.upcat_3_1 = UpCat(spatial_dims, F4, F3, F3, act, norm, True, dropout, upsample)

        self.upcat_0_2 = UpCat(spatial_dims, F1, F0*2, F0, act, norm, True, dropout, upsample, halves=False)
        self.upcat_1_2 = UpCat(spatial_dims, F2, F1*2, F1, act, norm, True, dropout, upsample)
        self.upcat_2_2 = UpCat(spatial_dims, F3, F2*2, F2, act, norm, True, dropout, upsample)

        self.upcat_0_3 = UpCat(spatial_dims, F1, F0*3, F0, act, norm, True, dropout, upsample, halves=False)
        self.upcat_1_3 = UpCat(spatial_dims, F2, F1*3, F1, act, norm, True, dropout, upsample)

        self.upcat_0_4 = UpCat(spatial_dims, F1, F0*4, Fhead, act, norm, True, dropout, upsample, halves=False)

        self.final_conv_0_1 = Conv["conv", spatial_dims](F0,    out_channels, kernel_size=1)
        self.final_conv_0_2 = Conv["conv", spatial_dims](F0,    out_channels, kernel_size=1)
        self.final_conv_0_3 = Conv["conv", spatial_dims](F0,    out_channels, kernel_size=1)
        self.final_conv_0_4 = Conv["conv", spatial_dims](Fhead, out_channels, kernel_size=1)

    def forward(self, x, feats):
        # feats dari encoder ResNet18: s0(~ /2), s1(~ /4), s2(~ /8), s3(~ /16)
        s0, s1, s2, s3 = feats
    
        # adapt channel
        s0 = self.enc_adapt1(s0)
        s1 = self.enc_adapt2(s1)
        s2 = self.enc_adapt3(s2)
        s3 = self.enc_adapt4(s3)
    
        x_0_0 = self.conv_0_0(x)         # /1 (dimensi sama dgn input)
        D, H, W = x_0_0.shape[2:]
        size_1  = (max(1, D // 2),  max(1, H // 2),  max(1, W // 2))   # /2
        size_2  = (max(1, D // 4),  max(1, H // 4),  max(1, W // 4))   # /4
        size_3  = (max(1, D // 8),  max(1, H // 8),  max(1, W // 8))   # /8
        size_4  = (max(1, D // 16), max(1, H // 16), max(1, W // 16))  # /16
    
        s0 = F.interpolate(s0, size=size_1, mode="trilinear", align_corners=False)
        s1 = F.interpolate(s1, size=size_2, mode="trilinear", align_corners=False)
        s2 = F.interpolate(s2, size=size_3, mode="trilinear", align_corners=False)
        s3 = F.interpolate(s3, size=size_4, mode="trilinear", align_corners=False)
    

        x_1_0 = s0
        x_0_1 = self.upcat_0_1(x_1_0, x_0_0)
    
        x_2_0 = s1
        x_1_1 = self.upcat_1_1(x_2_0, x_1_0)
        x_0_2 = self.upcat_0_2(x_1_1, torch.cat([x_0_0, x_0_1], dim=1))
    
        x_3_0 = s2
        x_2_1 = self.upcat_2_1(x_3_0, x_2_0)
        x_1_2 = self.upcat_1_2(x_2_1, torch.cat([x_1_0, x_1_1], dim=1))
        x_0_3 = self.upcat_0_3(x_1_2, torch.cat([x_0_0, x_0_1, x_0_2], dim=1))
    
        x_4_0 = s3
        x_3_1 = self.upcat_3_1(x_4_0, x_3_0)
        x_2_2 = self.upcat_2_2(x_3_1, torch.cat([x_2_0, x_2_1], dim=1))
        x_1_3 = self.upcat_1_3(x_2_2, torch.cat([x_1_0, x_1_1, x_1_2], dim=1))
        x_0_4 = self.upcat_0_4(x_1_3, torch.cat([x_0_0, x_0_1, x_0_2, x_0_3], dim=1))
    
        y_0_1 = self.final_conv_0_1(x_0_1)
        y_0_2 = self.final_conv_0_2(x_0_2)
        y_0_3 = self.final_conv_0_3(x_0_3)
        y_0_4 = self.final_conv_0_4(x_0_4)

        if self.deep_supervision:
            return [y_0_4, y_0_3, y_0_2, y_0_1] if self.training else y_0_4
        else:
            return y_0_4

class UNetPP3D_MedicalNet(nn.Module):
    def __init__(self,
        in_channels=4,
        out_channels=3,
        features=(16,32,64,128,128,16),
        enc_channels=(64,128,256,512),
    ):
        super().__init__()
        self.encoder = MedicalNetResNet18Encoder(in_channels=in_channels)
        self.decoder = UNetPP3D_Decoder(
            spatial_dims=3,
            in_channels=in_channels,
            out_channels=out_channels,
            features=features,
            enc_channels=enc_channels,
            upsample="nontrainable",
            deep_supervision=True,
        )

    def load_medicalnet(self, pth_path):
        self.encoder.load_medicalnet_pretrain(
            pth_path=pth_path,
            in_channels=self.encoder.backbone.conv1.in_channels,
        )

    def forward(self, x):
        feats = self.encoder(x)
        return self.decoder(x, feats)


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from monai.networks.layers.factories import Conv
from monai.networks.nets.basic_unet import TwoConv, UpCat
import monai.networks.nets.resnet as monai_resnet


def prepare_medicalnet_state_for_monai(sd_in, in_channels=4):
    sd_raw = sd_in.get("state_dict", sd_in)
    sd = dict(sd_raw)
    remap = {}

    for k, v in sd.items():
        kk = k[len("module."):] if k.startswith("module.") else k

        # drop classifier
        if kk.startswith("fc."):
            continue

        # strip inner ".conv" (varian yang simpan conv submodule)
        kk = kk.replace(".conv1.conv.", ".conv1.")
        kk = kk.replace(".conv2.conv.", ".conv2.")
        kk = kk.replace(".conv3.conv.", ".conv3.")  # aman utk R50 juga

        # map style 'downsample.conv/bn' → 'downsample.0/1'
        kk = kk.replace(".downsample.conv.", ".downsample.0.")
        kk = kk.replace(".downsample.bn.",  ".downsample.1.")

        # map style 'shortcut' → 'downsample'
        kk = kk.replace(".shortcut.0.", ".downsample.0.")
        kk = kk.replace(".shortcut.1.", ".downsample.1.")

        # sebagian ckpt pakai 'downsample.weight' (tanpa .0) → asumsi conv
        if ".downsample.weight" in kk and ".downsample.0.weight" not in kk:
            kk = kk.replace(".downsample.weight", ".downsample.0.weight")
        if ".downsample.bias" in kk and ".downsample.0.bias" not in kk:
            kk = kk.replace(".downsample.bias", ".downsample.0.bias")

        remap[kk] = v

    # adapt conv1 Cin=1 → multi-modal (mis. 4)
    for conv1_key in ["conv1.weight", "conv1.conv.weight"]:
        if conv1_key in remap:
            w = remap[conv1_key]
            if w.ndim == 5 and w.shape[1] == 1 and in_channels > 1:
                w = w.mean(1, keepdim=True).repeat(1, in_channels, 1, 1, 1) / float(in_channels)
                remap[conv1_key] = w
                break

    return remap


@torch.no_grad()
def load_medicalnet_into_monai_resnet(backbone, pth_path, in_channels=4, verbose=True):
    raw = torch.load(pth_path, map_location="cpu")
    sd  = prepare_medicalnet_state_for_monai(raw, in_channels=in_channels)
    info = backbone.load_state_dict(sd, strict=False)
    try:
        missing, unexpected = info.missing_keys, info.unexpected_keys
    except AttributeError:
        missing, unexpected = info
    if verbose:
        monai_keys = backbone.state_dict()
        matched = sum(int(k in monai_keys and monai_keys[k].shape == sd[k].shape) for k in sd.keys())
        print(f"[MedicalNet→MONAI] matched={matched}  missing={len(missing)}  unexpected={len(unexpected)}")
    return info


class MedicalNetResNet18Encoder(nn.Module):
    def __init__(self, in_channels=4):
        super().__init__()
        self.backbone = monai_resnet.resnet18(
            spatial_dims=3, n_input_channels=in_channels, pretrained=False, progress=False
        )
        self.layer1 = self.backbone.layer1
        self.layer2 = self.backbone.layer2
        self.layer3 = self.backbone.layer3
        self.layer4 = self.backbone.layer4

    @torch.no_grad()
    def load_medicalnet_pretrain(self, pth_path, in_channels=4):
        load_medicalnet_into_monai_resnet(
            backbone=self.backbone, pth_path=pth_path, in_channels=in_channels, verbose=True
        )

    def forward(self, x):
        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = F.relu(x, inplace=True)
        s0 = x                 # <<< /2 tap (SEBELUM maxpool)
        x  = self.backbone.maxpool(x)
        s1 = self.layer1(x)    # /4
        s2 = self.layer2(s1)   # /8
        s3 = self.layer3(s2)   # /16
        return s0, s1, s2, s3


class UNet3D_Decoder(nn.Module):
    def __init__(
        self,
        spatial_dims=3,
        in_channels=4,
        out_channels=3,
        features=(16, 32, 64, 128, 128),
        act=("LeakyReLU", {"negative_slope": 0.1, "inplace": True}),
        norm=("instance", {"affine": True}),
        dropout=0.0,
        upsample="nontrainable",
        enc_channels=(64, 128, 256, 512),
        deep_supervision=False,
    ):
        super().__init__()
        self.deep_supervision = deep_supervision
        F0, F1, F2, F3, F4 = features

        Conv1x1 = lambda a, b: Conv["conv", spatial_dims](a, b, kernel_size=1, bias=False)
        self.adapt_s0 = Conv1x1(enc_channels[0], F1)  # /2
        self.adapt_s1 = Conv1x1(enc_channels[1], F2)  # /4
        self.adapt_s2 = Conv1x1(enc_channels[2], F3)  # /8
        self.adapt_s3 = Conv1x1(enc_channels[3], F4)  # /16 (bottleneck in)

        self.conv_0_0 = TwoConv(spatial_dims, in_channels, F0, act, norm, bias=True, dropout=dropout)

        self.bottleneck = TwoConv(spatial_dims, F4, F4, act, norm, bias=True, dropout=dropout)


        self.up_3 = UpCat(spatial_dims, F4, F3, F3, act, norm, bias=True, dropout=dropout,
                           upsample=upsample)
        self.up_2 = UpCat(spatial_dims, F3, F2, F2, act, norm, bias=True, dropout=dropout,
                           upsample=upsample)
        self.up_1 = UpCat(spatial_dims, F2, F1, F1, act, norm, bias=True, dropout=dropout,
                           upsample=upsample)
        self.up_0 = UpCat(spatial_dims, F1, F0, F0, act, norm, bias=True, dropout=dropout,
                           upsample=upsample, halves=False)

        # Head
        self.head = Conv["conv", spatial_dims](F0, out_channels, kernel_size=1)

        # (opsional) heads untuk deep supervision jika ingin dihidupkan
        if self.deep_supervision:
            self.ds2 = Conv["conv", spatial_dims](F1, out_channels, kernel_size=1)  # /2
            self.ds3 = Conv["conv", spatial_dims](F2, out_channels, kernel_size=1)  # /4
            self.ds4 = Conv["conv", spatial_dims](F3, out_channels, kernel_size=1)  # /8

    def forward(self, x, feats):
        s0, s1, s2, s3 = feats

        # Normalisasi spasial → tepat /2,/4,/8,/16 dari ukuran input setelah conv_0_0
        x_0_0 = self.conv_0_0(x)   # /1
        D, H, W = x_0_0.shape[2:]
        size_1  = (max(1, D // 2),  max(1, H // 2),  max(1, W // 2))   # /2
        size_2  = (max(1, D // 4),  max(1, H // 4),  max(1, W // 4))   # /4
        size_3  = (max(1, D // 8),  max(1, H // 8),  max(1, W // 8))   # /8
        size_4  = (max(1, D // 16), max(1, H // 16), max(1, W // 16))  # /16

        # Adapt channel lalu *force* ke ukuran yang konsisten
        s0 = F.interpolate(self.adapt_s0(s0), size=size_1, mode="trilinear", align_corners=False)
        s1 = F.interpolate(self.adapt_s1(s1), size=size_2, mode="trilinear", align_corners=False)
        s2 = F.interpolate(self.adapt_s2(s2), size=size_3, mode="trilinear", align_corners=False)
        s3 = F.interpolate(self.adapt_s3(s3), size=size_4, mode="trilinear", align_corners=False)

        # Bottleneck (/16)
        x_3 = self.bottleneck(s3)

        # Naik tangga UNet
        x_2 = self.up_3(x_3, s2)              # /8
        x_1 = self.up_2(x_2, s1)              # /4
        x_0_5 = self.up_1(x_1, s0)            # /2
        x_out = self.up_0(x_0_5, x_0_0)       # /1

        y = self.head(x_out)

        if not self.deep_supervision:
            return y

        y_ds2 = self.ds2(x_0_5)  # /2
        y_ds3 = self.ds3(x_1)    # /4
        y_ds4 = self.ds4(x_2)    # /8
        # urutan: finest → coarser (selaras dgn pola Anda)
        return [y, y_ds2, y_ds3, y_ds4]


class UNet3D_MedicalNet(nn.Module):
    def __init__(
        self,
        in_channels=4,
        out_channels=3,
        features=(16, 32, 64, 128, 128),
        enc_channels=(64, 128, 256, 512),
        deep_supervision=False,
    ):
        super().__init__()
        self.encoder = MedicalNetResNet18Encoder(in_channels=in_channels)
        self.decoder = UNet3D_Decoder(
            spatial_dims=3,
            in_channels=in_channels,
            out_channels=out_channels,
            features=features,
            enc_channels=enc_channels,
            upsample="nontrainable",
            deep_supervision=deep_supervision,
        )

    def load_medicalnet(self, pth_path):
        self.encoder.load_medicalnet_pretrain(
            pth_path=pth_path,
            in_channels=self.encoder.backbone.conv1.in_channels,
        )

    def forward(self, x):
        feats = self.encoder(x)      # (s0, s1, s2, s3)
        return self.decoder(x, feats)


# Import data

In [ ]:
import os
from pathlib import Path
MAIN_DIR = str(SAMPLE_DIR)


def get_data_paths(data_dir):
    suffixes = ["flair", "t1", "t1ce", "t2", "seg"]

    def strip_ext(fn):
        if fn.endswith(".nii.gz"): return fn[:-7]
        if fn.endswith(".nii"):    return fn[:-4]
        return fn

    def split_pid_suffix(basename_no_ext):
        # kembalikan (pid, suffix) jika nama berakhir dengan _suffix
        for s in suffixes:
            tag = "_" + s
            if basename_no_ext.endswith(tag):
                return basename_no_ext[:-len(tag)], s
        return None, None

    data_paths = []

    # Cek apakah di dalam data_dir ada subfolder pasien
    subdirs = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]

    if subdirs:
        # Mode: per-pasien folder
        for d in sorted(subdirs):
            folder = os.path.join(data_dir, d)
            bucket = {}
            for f in os.listdir(folder):
                if not (f.endswith(".nii") or f.endswith(".nii.gz")): continue
                pid, sfx = split_pid_suffix(strip_ext(f))
                if sfx: bucket[sfx] = os.path.join(folder, f)
            if all(k in bucket for k in suffixes):
                # ambil pid dari salah satu file (flair)
                pid, _ = split_pid_suffix(strip_ext(os.path.basename(bucket["flair"])))
                data_paths.append({
                    "flair": bucket["flair"],
                    "t1":    bucket["t1"],
                    "t1gd":  bucket["t1ce"],   # konsistensi key t1gd=t1ce
                    "t2":    bucket["t2"],
                    "label": bucket["seg"],
                    "pid":   pid
                })
    else:
        # Mode: flat (semua file langsung di data_dir)
        files = [f for f in os.listdir(data_dir) if f.endswith(".nii") or f.endswith(".nii.gz")]
        groups = {}
        for f in files:
            pid, sfx = split_pid_suffix(strip_ext(f))
            if pid is None: continue
            groups.setdefault(pid, {})[sfx] = os.path.join(data_dir, f)

        for pid, g in sorted(groups.items()):
            if all(k in g for k in suffixes):
                data_paths.append({
                    "flair": g["flair"],
                    "t1":    g["t1"],
                    "t1gd":  g["t1ce"],
                    "t2":    g["t2"],
                    "label": g["seg"],
                    "pid":   pid
                })

    print(f"Found {len(data_paths)} samples in {data_dir}.")
    if not data_paths:
        print("Hint: pastikan MAIN_DIR menunjuk langsung ke folder yang berisi file NIfTI (mis. data/samples/FeTS21_Training_001)")
    return data_paths


all_paths = get_data_paths(MAIN_DIR)
if not all_paths:
    raise RuntimeError("No data found. Check MAIN_DIR & filenames.")


In [ ]:
from monai.transforms import (MapTransform)

class BraTSLabelToETTCWT(MapTransform):
    """
    Input d["label"] shape [1,D,H,W] atau [D,H,W] berisi {0,1,2,4}.
    Output d["label_ettcwt"] shape [3,D,H,W]: [ET, TC, WT] sebagai mask biner.
    """
    def __init__(self, keys=("label",), out_key="label_ettcwt"):
        super().__init__(keys); self.out_key = out_key

    def __call__(self, data):
        d = dict(data)
        lab = d[self.keys[0]]
        lab = torch.as_tensor(lab)
        if lab.ndim == 4 and lab.shape[0] == 1: lab = lab[0]  # [D,H,W]
        et = (lab == 4).float()
        tc = ((lab == 1) | (lab == 4)).float()
        wt = ((lab == 1) | (lab == 2) | (lab == 4)).float()
        d[self.out_key] = torch.stack([et, tc, wt], dim=0)  # [3,D,H,W]
        return d

In [ ]:
from monai.transforms import (
    LoadImaged, EnsureChannelFirstd, EnsureTyped, Compose, SelectItemsd,
    NormalizeIntensityd, CropForegroundd, RandSpatialCropd,
    RandScaleIntensityd, RandShiftIntensityd, RandFlipd, ConcatItemsd, MapTransform
)

test_transforms = Compose([
    LoadImaged(keys=["flair","t1","t1gd","t2","label"]),
    EnsureChannelFirstd(keys=["flair","t1","t1gd","t2","label"]),
    ConcatItemsd(keys=["flair","t1","t1gd","t2"], name="image", dim=0),
    EnsureTyped(keys=["image","label"], track_meta=False),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    CropForegroundd(keys=["image","label"], source_key="image"),
    RandSpatialCropd(keys=["image","label"], roi_size=(128,128,128), random_center=False, random_size=False),
    BraTSLabelToETTCWT(keys=("label",), out_key="label_ettcwt"),
    SelectItemsd(keys=["image","label_ettcwt","pid"]),
])

# Predict E1

In [ ]:
# ==== 1) Setup: pilih model & checkpoint ====
import os, math, torch, numpy as np, nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path
from monai.inferers import sliding_window_inference
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_TYPE   = "unetpp"   # "unetpp" atau "unet"
CKPT_PATH = configured_path("BTS_UNETPP_LR_CHECKPOINT", CHECKPOINT_ROOT / "unetpp_warmup_lr" / "unetpp_medicalnet_multilabel_best_20251012-221130.pt")
THRESH       = 0.5        # sigmoid threshold untuk ET/TC/WT
ROI_DEFAULT  = (128,128,128)  # fallback roi (akan di-cap by volume size)

if MODEL_TYPE == "unetpp":
    model = UNetPP3D_MedicalNet(
        in_channels=4, out_channels=3,
        features=(16,32,64,128,128,16),
        enc_channels=(64,64,128,256),
    ).to(device)
elif MODEL_TYPE == "unet":
    # DS ON agar cocok dengan checkpoint training-mu
    model = UNet3D_MedicalNet(
        in_channels=4, out_channels=3,
        features=(16,32,64,128,128),
        enc_channels=(64,64,128,256),
        deep_supervision=True,   # <--- ON (penting)
    ).to(device)
else:
    raise ValueError("MODEL_TYPE harus 'unetpp' atau 'unet'.")

state = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(state, strict=True)
model.eval()
for m in model.modules():
    if isinstance(m, (torch.nn.BatchNorm3d, torch.nn.SyncBatchNorm)):
        m.eval()

# ==== 6) Predictor ENSAMBLE (Opsi B) ====
# Bekerja untuk:
# - UNet biasa DS ON → out = [y_full, y_/2, y_/4, y_/8] → upsample & rata-rata
# - UNet++ eval → out tensor tunggal → langsung return
def predictor_ensemble(x, weights=(1.0, 0.5, 0.25, 0.125)):
    outs = model(x)
    if not isinstance(outs, (list, tuple)):
        return outs  # mis. UNet++ eval mengembalikan single head
    # asumsi urutan: [y_full(/1), y_ds2(/2), y_ds3(/4), y_ds4(/8)]
    y0, y2, y3, y4 = outs
    tgt = y0.shape[2:]  # (D,H,W)
    y2u = F.interpolate(y2, size=tgt, mode="trilinear", align_corners=False)
    y3u = F.interpolate(y3, size=tgt, mode="trilinear", align_corners=False)
    y4u = F.interpolate(y4, size=tgt, mode="trilinear", align_corners=False)
    w = torch.tensor(weights, dtype=y0.dtype, device=y0.device)
    if w.numel() != 4:
        raise ValueError("weights harus berisi 4 angka untuk [y0, y/2, y/4, y/8].")
    w = w / w.sum()
    return w[0]*y0 + w[1]*y2u + w[2]*y3u + w[3]*y4u

# ==== 7) Ambil satu kasus & apply transforms ====
idx = 0
sample_raw = all_paths[idx]
sample_t   = test_transforms(sample_raw)
img = sample_t["image"].unsqueeze(0).to(device)  # [1,4,D,H,W]
pid = sample_t["pid"]

# ==== 8) Sliding-window inference (pakai predictor_ensemble) ====
_, _, D, H, W = img.shape
roi = (min(ROI_DEFAULT[0], D), min(ROI_DEFAULT[1], H), min(ROI_DEFAULT[2], W))
with torch.no_grad():
    logits = sliding_window_inference(
        inputs=img, roi_size=roi, sw_batch_size=1,
        predictor=predictor_ensemble, overlap=0.5, mode="gaussian"
    )  # [1,3,D,H,W]

probs = torch.sigmoid(logits)[0].cpu().numpy()  # (3,D,H,W)
pred  = (probs > THRESH).astype(np.uint8)

# --- sanity check ---
print("prob stats: min/max/mean =", float(probs.min()), float(probs.max()), float(probs.mean()))
voxel_counts = pred.reshape(3, -1).sum(1)
print("voxel counts  ET/TC/WT  =", voxel_counts.tolist())

# per-slice (axial) jumlah voxel bertopeng
per_slice = pred.any(axis=0).sum(axis=(1,2))
print("top slices by prediction:", np.argsort(-per_slice)[:10])


# ==== 9) Pilih irisan informatif ====
tumor_map = pred.any(axis=0)            # (D,H,W)
z_scores  = tumor_map.sum(axis=(1,2))   # (D,)
if z_scores.max() == 0:
    zs = np.linspace(0, D-1, 6, dtype=int)
else:
    zs = np.sort(np.argsort(-z_scores)[:6])

# ==== 10) Visualisasi overlay (FLAIR) ====
flair = img[0,0].cpu().numpy()
ET, TC, WT = pred[0], pred[1], pred[2]



def pick_best_slice(flair, pred=None, gt=None, prefer="gt"):
    """
    Pilih irisan (z) yang paling 'ramai' tumor.
    Urutan preferensi: GT -> Pred -> tengah volume.
    """
    z = None
    if prefer == "gt" and gt is not None:
        uni = gt.any(axis=0)                      # (D,H,W)
        sc  = uni.sum(axis=(1,2))                 # (D,)
        if sc.max() > 0:
            z = int(sc.argmax())
    if z is None and pred is not None:
        uni = pred.any(axis=0)
        sc  = uni.sum(axis=(1,2))
        if sc.max() > 0:
            z = int(sc.argmax())
    if z is None:
        z = flair.shape[0] // 2
    return z

def show_pred_gt_single(flair, pred, gt=None, z=None, mode="overlay", title=None, save_path=None):
    """
    mode='overlay'  -> 1 gambar, Pred (solid) + GT (putus-putus) di axis yang sama.
    mode='split'    -> 2 gambar berdampingan: kiri=Pred, kanan=GT.
    """
    # Siapkan slice index
    if z is None:
        z = pick_best_slice(flair, pred=pred, gt=gt, prefer="gt")  # pilih otomatis

    lo, hi = np.percentile(flair, 1), np.percentile(flair, 99)
    ETp, TCp, WTp = pred[0], pred[1], pred[2]
    ETg = TCg = WTg = None
    if gt is not None:
        ETg, TCg, WTg = gt[0], gt[1], gt[2]

    if mode == "split":
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axp, axg = axes

        # ---- PRED (kiri)
        axp.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        axp.imshow(np.ma.masked_where(ETp[z]==0, ETp[z]), cmap="Reds",   alpha=0.25, vmin=0, vmax=1)
        axp.imshow(np.ma.masked_where(TCp[z]==0, TCp[z]), cmap="Greens", alpha=0.25, vmin=0, vmax=1)
        axp.imshow(np.ma.masked_where(WTp[z]==0, WTp[z]), cmap="Blues",  alpha=0.25, vmin=0, vmax=1)
        if ETp[z].any(): axp.contour(ETp[z], levels=[0.5], colors="#ff2d2d", linewidths=2.0)
        if TCp[z].any(): axp.contour(TCp[z], levels=[0.5], colors="#2dcc2d", linewidths=2.0)
        if WTp[z].any(): axp.contour(WTp[z], levels=[0.5], colors="#2d4dff", linewidths=2.0)
        axp.set_title(f"z={z} — Prediction")
        axp.axis("off")

        # ---- GT (kanan)
        axg.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        if gt is not None:
            axg.imshow(np.ma.masked_where(ETg[z]==0, ETg[z]), cmap="Reds",   alpha=0.25, vmin=0, vmax=1)
            axg.imshow(np.ma.masked_where(TCg[z]==0, TCg[z]), cmap="Greens", alpha=0.25, vmin=0, vmax=1)
            axg.imshow(np.ma.masked_where(WTg[z]==0, WTg[z]), cmap="Blues",  alpha=0.25, vmin=0, vmax=1)
            if ETg[z].any(): axg.contour(ETg[z], levels=[0.5], colors="#ff2d2d", linewidths=2, linestyles="--")
            if TCg[z].any(): axg.contour(TCg[z], levels=[0.5], colors="#2dcc2d", linewidths=2, linestyles="--")
            if WTg[z].any(): axg.contour(WTg[z], levels=[0.5], colors="#2d4dff", linewidths=2, linestyles="--")
        axg.set_title(f"z={z} — Ground Truth")
        axg.axis("off")

        if title: fig.suptitle(title, fontsize=14)
        plt.tight_layout()
        if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()

    else:  # mode == "overlay"
        fig, ax = plt.subplots(1, 1, figsize=(6, 6))
        ax.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        # Pred (solid)
        if ETp[z].any(): ax.contour(ETp[z], levels=[0.5], colors="#ff2d2d", linewidths=2.0)
        if TCp[z].any(): ax.contour(TCp[z], levels=[0.5], colors="#2dcc2d", linewidths=2.0)
        if WTp[z].any(): ax.contour(WTp[z], levels=[0.5], colors="#2d4dff", linewidths=2.0)
        # GT (dashed)
        if gt is not None:
            if ETg[z].any(): ax.contour(ETg[z], levels=[0.5], colors="#ff2d2d", linewidths=2.0, linestyles="--")
            if TCg[z].any(): ax.contour(TCg[z], levels=[0.5], colors="#2dcc2d", linewidths=2.0, linestyles="--")
            if WTg[z].any(): ax.contour(WTg[z], levels=[0.5], colors="#2d4dff", linewidths=2.0, linestyles="--")

        ax.set_title(f"z={z} — Pred (solid) vs GT (dashed)")
        ax.axis("off")
        if title: fig.suptitle(title, fontsize=14)
        plt.tight_layout()
        if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()

    print(f"[INFO] Menampilkan slice z={z}")
    return z

def show_overlay_grid(z_indices, cols=3, title=None, save_path=None):
    rows = math.ceil(len(z_indices)/cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
    axes = np.array(axes).reshape(rows, cols)

    # normalisasi window supaya kontras
    lo, hi = np.percentile(flair, 1), np.percentile(flair, 99)

    for i, z in enumerate(z_indices):
        r, c = divmod(i, cols)
        ax = axes[r, c]
        ax.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)

        # kontur tebal
        if ET[z].any():
            ax.contour(ET[z], levels=[0.5], colors='#ff2d2d', linewidths=2.0)
        if TC[z].any():
            ax.contour(TC[z], levels=[0.5], colors='#2dcc2d', linewidths=2.0)
        if WT[z].any():
            ax.contour(WT[z], levels=[0.5], colors='#2d4dff', linewidths=2.0)

        # overlay fill tipis (opsional)
        et_fill = np.ma.masked_where(ET[z]==0, ET[z])
        tc_fill = np.ma.masked_where(TC[z]==0, TC[z])
        wt_fill = np.ma.masked_where(WT[z]==0, WT[z])
        ax.imshow(et_fill, cmap='Reds',   alpha=0.25, vmin=0, vmax=1)
        ax.imshow(tc_fill, cmap='Greens', alpha=0.25, vmin=0, vmax=1)
        ax.imshow(wt_fill, cmap='Blues',  alpha=0.25, vmin=0, vmax=1)

        ax.set_title(f"z={z} | ET red, TC green, WT blue")
        ax.axis("off")

    from matplotlib.patches import Patch
    handles = [Patch(color='#ff2d2d', label='ET'),
               Patch(color='#2dcc2d', label='TC'),
               Patch(color='#2d4dff', label='WT')]
    if title: fig.suptitle(title, fontsize=14)
    fig.legend(handles=handles, loc='upper right')
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

def show_gt_overlay(z_indices, flair, gt, cols=3, title=None, save_path=None):
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch
    import math

    ETg, TCg, WTg = gt[0], gt[1], gt[2]
    rows = math.ceil(len(z_indices)/cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
    axes = np.array(axes).reshape(rows, cols)
    lo, hi = np.percentile(flair, 1), np.percentile(flair, 99)

    for i, z in enumerate(z_indices):
        r, c = divmod(i, cols)
        ax = axes[r, c]
        ax.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        # fill tipis
        ax.imshow(np.ma.masked_where(ETg[z]==0, ETg[z]), cmap='Reds',   alpha=0.25, vmin=0, vmax=1)
        ax.imshow(np.ma.masked_where(TCg[z]==0, TCg[z]), cmap='Greens', alpha=0.25, vmin=0, vmax=1)
        ax.imshow(np.ma.masked_where(WTg[z]==0, WTg[z]), cmap='Blues',  alpha=0.25, vmin=0, vmax=1)
        # kontur putus-putus
        if ETg[z].any(): ax.contour(ETg[z], levels=[0.5], colors='#ff2d2d', linewidths=2, linestyles='--')
        if TCg[z].any(): ax.contour(TCg[z], levels=[0.5], colors='#2dcc2d', linewidths=2, linestyles='--')
        if WTg[z].any(): ax.contour(WTg[z], levels=[0.5], colors='#2d4dff', linewidths=2, linestyles='--')
        ax.set_title(f"z={z} | GT")
        ax.axis("off")

    handles = [Patch(color='#ff2d2d', label='ET (GT)'),
               Patch(color='#2dcc2d', label='TC (GT)'),
               Patch(color='#2d4dff', label='WT (GT)')]
    if title: fig.suptitle(title, fontsize=14)
    fig.legend(handles=handles, loc='upper right')
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


def show_pred_vs_gt(z_indices, flair, pred, gt, title=None, save_path=None):
    """Grid 2 kolom per slice: kiri = Pred, kanan = GT."""
    import numpy as np
    import matplotlib.pyplot as plt
    import math

    ETp, TCp, WTp = pred[0], pred[1], pred[2]
    ETg, TCg, WTg = gt[0],   gt[1],   gt[2]

    rows, cols = len(z_indices), 2
    fig, axes = plt.subplots(rows, cols, figsize=(8*cols, 4*rows))
    lo, hi = np.percentile(flair, 1), np.percentile(flair, 99)

    for i, z in enumerate(z_indices):
        # ---- Pred (kiri)
        axp = axes[i, 0]
        axp.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        axp.imshow(np.ma.masked_where(ETp[z]==0, ETp[z]), cmap='Reds',   alpha=0.25, vmin=0, vmax=1)
        axp.imshow(np.ma.masked_where(TCp[z]==0, TCp[z]), cmap='Greens', alpha=0.25, vmin=0, vmax=1)
        axp.imshow(np.ma.masked_where(WTp[z]==0, WTp[z]), cmap='Blues',  alpha=0.25, vmin=0, vmax=1)
        if ETp[z].any(): axp.contour(ETp[z], levels=[0.5], colors='#ff2d2d', linewidths=2.0)
        if TCp[z].any(): axp.contour(TCp[z], levels=[0.5], colors='#2dcc2d', linewidths=2.0)
        if WTp[z].any(): axp.contour(WTp[z], levels=[0.5], colors='#2d4dff', linewidths=2.0)
        axp.set_title(f"z={z} — Pred")
        axp.axis("off")

        # ---- GT (kanan)
        axg = axes[i, 1]
        axg.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        axg.imshow(np.ma.masked_where(ETg[z]==0, ETg[z]), cmap='Reds',   alpha=0.25, vmin=0, vmax=1)
        axg.imshow(np.ma.masked_where(TCg[z]==0, TCg[z]), cmap='Greens', alpha=0.25, vmin=0, vmax=1)
        axg.imshow(np.ma.masked_where(WTg[z]==0, WTg[z]), cmap='Blues',  alpha=0.25, vmin=0, vmax=1)
        if ETg[z].any(): axg.contour(ETg[z], levels=[0.5], colors='#ff2d2d', linewidths=2, linestyles='--')
        if TCg[z].any(): axg.contour(TCg[z], levels=[0.5], colors='#2dcc2d', linewidths=2, linestyles='--')
        if WTg[z].any(): axg.contour(WTg[z], levels=[0.5], colors='#2d4dff', linewidths=2, linestyles='--')
        axg.set_title(f"z={z} — GT")
        axg.axis("off")

    if title: fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    
def show_pred_vs_gt_pair(flair, pred, gt, z=None, title=None, save_path=None,
                         fill_alpha=0.25, prefer="gt"):
    """
    Tampilkan 1 slice berdampingan: kiri = Prediksi, kanan = Ground Truth (SEG).
    - flair: (D,H,W) float
    - pred : (3,D,H,W) uint8/bool  -> [ET,TC,WT]
    - gt   : (3,D,H,W) uint8/bool  -> [ET,TC,WT]
    - z    : index slice (opsional); jika None dipilih otomatis (prioritas GT)
    """
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch

    # --- validasi bentuk ---
    assert pred.ndim == 4 and pred.shape[0] == 3, "pred harus (3,D,H,W)"
    assert gt.ndim   == 4 and gt.shape[0]   == 3, "gt harus (3,D,H,W)"
    D, H, W = flair.shape
    assert pred.shape[1:] == (D, H, W), "pred & flair harus ukuran sama"
    assert gt.shape[1:]   == (D, H, W), "gt & flair harus ukuran sama"

    # --- pilih slice otomatis bila perlu ---
    def pick_best_slice(_pred, _gt, prefer="gt"):
        cand = None
        if prefer == "gt" and _gt is not None and _gt.any():
            sc = _gt.any(axis=0).sum(axis=(1,2))  # (D,)
            if sc.max() > 0: return int(sc.argmax())
        if _pred is not None and _pred.any():
            sc = _pred.any(axis=0).sum(axis=(1,2))
            if sc.max() > 0: return int(sc.argmax())
        return D // 2

    if z is None:
        z = pick_best_slice(pred, gt, prefer=prefer)

    # --- windowing biar kontras enak ---
    lo, hi = np.percentile(flair, 1), np.percentile(flair, 99)

    ETp, TCp, WTp = pred[0], pred[1], pred[2]
    ETg, TCg, WTg = gt[0],   gt[1],   gt[2]

    fig, (axp, axg) = plt.subplots(1, 2, figsize=(12, 5))

    # ===== KIRI: PREDIKSI =====
    axp.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
    axp.imshow(np.ma.masked_where(ETp[z]==0, ETp[z]), cmap="Reds",   alpha=fill_alpha, vmin=0, vmax=1)
    axp.imshow(np.ma.masked_where(TCp[z]==0, TCp[z]), cmap="Greens", alpha=fill_alpha, vmin=0, vmax=1)
    axp.imshow(np.ma.masked_where(WTp[z]==0, WTp[z]), cmap="Blues",  alpha=fill_alpha, vmin=0, vmax=1)
    if ETp[z].any(): axp.contour(ETp[z], levels=[0.5], colors="#ff2d2d", linewidths=2.0)
    if TCp[z].any(): axp.contour(TCp[z], levels=[0.5], colors="#2dcc2d", linewidths=2.0)
    if WTp[z].any(): axp.contour(WTp[z], levels=[0.5], colors="#2d4dff", linewidths=2.0)
    axp.set_title(f"z={z} — Prediction")
    axp.axis("off")

    # ===== KANAN: GROUND TRUTH =====
    axg.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
    axg.imshow(np.ma.masked_where(ETg[z]==0, ETg[z]), cmap="Reds",   alpha=fill_alpha, vmin=0, vmax=1)
    axg.imshow(np.ma.masked_where(TCg[z]==0, TCg[z]), cmap="Greens", alpha=fill_alpha, vmin=0, vmax=1)
    axg.imshow(np.ma.masked_where(WTg[z]==0, WTg[z]), cmap="Blues",  alpha=fill_alpha, vmin=0, vmax=1)
    if ETg[z].any(): axg.contour(ETg[z], levels=[0.5], colors="#ff2d2d", linewidths=2.0, linestyles="--")
    if TCg[z].any(): axg.contour(TCg[z], levels=[0.5], colors="#2dcc2d", linewidths=2.0, linestyles="--")
    if WTg[z].any(): axg.contour(WTg[z], levels=[0.5], colors="#2d4dff", linewidths=2.0, linestyles="--")
    axg.set_title(f"z={z} — Ground Truth")
    axg.axis("off")

    # legend
    handles = [Patch(color="#ff2d2d", label="ET"),
               Patch(color="#2dcc2d", label="TC"),
               Patch(color="#2d4dff", label="WT")]
    if title: fig.suptitle(title, fontsize=14)
    fig.legend(handles=handles, loc="upper right")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    return z

# Pred-only (sudah ada)
show_overlay_grid(zs, cols=3, title=f"{pid} — {MODEL_TYPE.upper()} (DS Ensemble)")

# Siapkan GT dari sample_t (sudah ter-transform & align dengan img)
gt_t = sample_t["label_ettcwt"]
gt = gt_t.cpu().numpy().astype(np.uint8) if torch.is_tensor(gt_t) else gt_t.astype(np.uint8)

# Pilih slice GT (atau pakai zs supaya slice-nya sama persis dengan pred)
gt_union = gt.any(axis=0)
z_scores_gt = gt_union.sum(axis=(1,2))
zs_gt = np.linspace(0, gt.shape[1]-1, 6, dtype=int) if z_scores_gt.max()==0 else np.sort(np.argsort(-z_scores_gt)[:6])

# GT-only overlay
show_gt_overlay(zs_gt, flair=flair, gt=gt, cols=3, title=f"{pid} — Ground Truth (SEG)")

gt_t = sample_t["label_ettcwt"]
gt   = gt_t.cpu().numpy().astype(np.uint8) if torch.is_tensor(gt_t) else gt_t.astype(np.uint8)

_ = show_pred_vs_gt_pair(
    flair=flair,
    pred=pred,
    gt=gt,
    z=None,  # biarkan auto-pick slice paling informatif
    title=f"{pid} — {MODEL_TYPE.upper()} (Pred vs SEG)",
    save_path=None  # isi path kalau mau disimpan
)

gt_t = sample_t["label_ettcwt"]
gt = gt_t.cpu().numpy().astype(np.uint8) if torch.is_tensor(gt_t) else gt_t.astype(np.uint8)

# Tampilkan satu irisan, overlay Pred vs GT di satu gambar
_ = show_pred_gt_single(flair, pred, gt=gt, z=None, mode="overlay",
                        title=f"{pid} — {MODEL_TYPE.upper()} (1 slice)")


# Predict E2

In [ ]:
# ==== 1) Setup: pilih model & checkpoint ====
import os, math, torch, numpy as np, nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path
from monai.inferers import sliding_window_inference
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_TYPE   = "unetpp"   # "unetpp" atau "unet"
CKPT_PATH = configured_path("BTS_UNETPP_FREEZE_CHECKPOINT", CHECKPOINT_ROOT / "unetpp_warmup_freeze" / "unetpp_medicalnet_multilabel_best_20251013-224841.pt")
THRESH       = 0.5        # sigmoid threshold untuk ET/TC/WT
ROI_DEFAULT  = (128,128,128)  # fallback roi (akan di-cap by volume size)

if MODEL_TYPE == "unetpp":
    model = UNetPP3D_MedicalNet(
        in_channels=4, out_channels=3,
        features=(16,32,64,128,128,16),
        enc_channels=(64,64,128,256),
    ).to(device)
elif MODEL_TYPE == "unet":
    # DS ON agar cocok dengan checkpoint training-mu
    model = UNet3D_MedicalNet(
        in_channels=4, out_channels=3,
        features=(16,32,64,128,128),
        enc_channels=(64,64,128,256),
        deep_supervision=True,   # <--- ON (penting)
    ).to(device)
else:
    raise ValueError("MODEL_TYPE harus 'unetpp' atau 'unet'.")

state = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(state, strict=True)
model.eval()
for m in model.modules():
    if isinstance(m, (torch.nn.BatchNorm3d, torch.nn.SyncBatchNorm)):
        m.eval()

# ==== 6) Predictor ENSAMBLE (Opsi B) ====
# Bekerja untuk:
# - UNet biasa DS ON → out = [y_full, y_/2, y_/4, y_/8] → upsample & rata-rata
# - UNet++ eval → out tensor tunggal → langsung return
def predictor_ensemble(x, weights=(1.0, 0.5, 0.25, 0.125)):
    outs = model(x)
    if not isinstance(outs, (list, tuple)):
        return outs  # mis. UNet++ eval mengembalikan single head
    # asumsi urutan: [y_full(/1), y_ds2(/2), y_ds3(/4), y_ds4(/8)]
    y0, y2, y3, y4 = outs
    tgt = y0.shape[2:]  # (D,H,W)
    y2u = F.interpolate(y2, size=tgt, mode="trilinear", align_corners=False)
    y3u = F.interpolate(y3, size=tgt, mode="trilinear", align_corners=False)
    y4u = F.interpolate(y4, size=tgt, mode="trilinear", align_corners=False)
    w = torch.tensor(weights, dtype=y0.dtype, device=y0.device)
    if w.numel() != 4:
        raise ValueError("weights harus berisi 4 angka untuk [y0, y/2, y/4, y/8].")
    w = w / w.sum()
    return w[0]*y0 + w[1]*y2u + w[2]*y3u + w[3]*y4u

# ==== 7) Ambil satu kasus & apply transforms ====
idx = 0
sample_raw = all_paths[idx]
sample_t   = test_transforms(sample_raw)
img = sample_t["image"].unsqueeze(0).to(device)  # [1,4,D,H,W]
pid = sample_t["pid"]

# ==== 8) Sliding-window inference (pakai predictor_ensemble) ====
_, _, D, H, W = img.shape
roi = (min(ROI_DEFAULT[0], D), min(ROI_DEFAULT[1], H), min(ROI_DEFAULT[2], W))
with torch.no_grad():
    logits = sliding_window_inference(
        inputs=img, roi_size=roi, sw_batch_size=1,
        predictor=predictor_ensemble, overlap=0.5, mode="gaussian"
    )  # [1,3,D,H,W]

probs = torch.sigmoid(logits)[0].cpu().numpy()  # (3,D,H,W)
pred  = (probs > THRESH).astype(np.uint8)

# --- sanity check ---
print("prob stats: min/max/mean =", float(probs.min()), float(probs.max()), float(probs.mean()))
voxel_counts = pred.reshape(3, -1).sum(1)
print("voxel counts  ET/TC/WT  =", voxel_counts.tolist())

# per-slice (axial) jumlah voxel bertopeng
per_slice = pred.any(axis=0).sum(axis=(1,2))
print("top slices by prediction:", np.argsort(-per_slice)[:10])


# ==== 9) Pilih irisan informatif ====
tumor_map = pred.any(axis=0)            # (D,H,W)
z_scores  = tumor_map.sum(axis=(1,2))   # (D,)
if z_scores.max() == 0:
    zs = np.linspace(0, D-1, 6, dtype=int)
else:
    zs = np.sort(np.argsort(-z_scores)[:6])

# ==== 10) Visualisasi overlay (FLAIR) ====
flair = img[0,0].cpu().numpy()
ET, TC, WT = pred[0], pred[1], pred[2]



def pick_best_slice(flair, pred=None, gt=None, prefer="gt"):
    """
    Pilih irisan (z) yang paling 'ramai' tumor.
    Urutan preferensi: GT -> Pred -> tengah volume.
    """
    z = None
    if prefer == "gt" and gt is not None:
        uni = gt.any(axis=0)                      # (D,H,W)
        sc  = uni.sum(axis=(1,2))                 # (D,)
        if sc.max() > 0:
            z = int(sc.argmax())
    if z is None and pred is not None:
        uni = pred.any(axis=0)
        sc  = uni.sum(axis=(1,2))
        if sc.max() > 0:
            z = int(sc.argmax())
    if z is None:
        z = flair.shape[0] // 2
    return z

def show_pred_gt_single(flair, pred, gt=None, z=None, mode="overlay", title=None, save_path=None):
    """
    mode='overlay'  -> 1 gambar, Pred (solid) + GT (putus-putus) di axis yang sama.
    mode='split'    -> 2 gambar berdampingan: kiri=Pred, kanan=GT.
    """
    # Siapkan slice index
    if z is None:
        z = pick_best_slice(flair, pred=pred, gt=gt, prefer="gt")  # pilih otomatis

    lo, hi = np.percentile(flair, 1), np.percentile(flair, 99)
    ETp, TCp, WTp = pred[0], pred[1], pred[2]
    ETg = TCg = WTg = None
    if gt is not None:
        ETg, TCg, WTg = gt[0], gt[1], gt[2]

    if mode == "split":
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axp, axg = axes

        # ---- PRED (kiri)
        axp.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        axp.imshow(np.ma.masked_where(ETp[z]==0, ETp[z]), cmap="Reds",   alpha=0.25, vmin=0, vmax=1)
        axp.imshow(np.ma.masked_where(TCp[z]==0, TCp[z]), cmap="Greens", alpha=0.25, vmin=0, vmax=1)
        axp.imshow(np.ma.masked_where(WTp[z]==0, WTp[z]), cmap="Blues",  alpha=0.25, vmin=0, vmax=1)
        if ETp[z].any(): axp.contour(ETp[z], levels=[0.5], colors="#ff2d2d", linewidths=2.0)
        if TCp[z].any(): axp.contour(TCp[z], levels=[0.5], colors="#2dcc2d", linewidths=2.0)
        if WTp[z].any(): axp.contour(WTp[z], levels=[0.5], colors="#2d4dff", linewidths=2.0)
        axp.set_title(f"z={z} — Prediction")
        axp.axis("off")

        # ---- GT (kanan)
        axg.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        if gt is not None:
            axg.imshow(np.ma.masked_where(ETg[z]==0, ETg[z]), cmap="Reds",   alpha=0.25, vmin=0, vmax=1)
            axg.imshow(np.ma.masked_where(TCg[z]==0, TCg[z]), cmap="Greens", alpha=0.25, vmin=0, vmax=1)
            axg.imshow(np.ma.masked_where(WTg[z]==0, WTg[z]), cmap="Blues",  alpha=0.25, vmin=0, vmax=1)
            if ETg[z].any(): axg.contour(ETg[z], levels=[0.5], colors="#ff2d2d", linewidths=2, linestyles="--")
            if TCg[z].any(): axg.contour(TCg[z], levels=[0.5], colors="#2dcc2d", linewidths=2, linestyles="--")
            if WTg[z].any(): axg.contour(WTg[z], levels=[0.5], colors="#2d4dff", linewidths=2, linestyles="--")
        axg.set_title(f"z={z} — Ground Truth")
        axg.axis("off")

        if title: fig.suptitle(title, fontsize=14)
        plt.tight_layout()
        if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()

    else:  # mode == "overlay"
        fig, ax = plt.subplots(1, 1, figsize=(6, 6))
        ax.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        # Pred (solid)
        if ETp[z].any(): ax.contour(ETp[z], levels=[0.5], colors="#ff2d2d", linewidths=2.0)
        if TCp[z].any(): ax.contour(TCp[z], levels=[0.5], colors="#2dcc2d", linewidths=2.0)
        if WTp[z].any(): ax.contour(WTp[z], levels=[0.5], colors="#2d4dff", linewidths=2.0)
        # GT (dashed)
        if gt is not None:
            if ETg[z].any(): ax.contour(ETg[z], levels=[0.5], colors="#ff2d2d", linewidths=2.0, linestyles="--")
            if TCg[z].any(): ax.contour(TCg[z], levels=[0.5], colors="#2dcc2d", linewidths=2.0, linestyles="--")
            if WTg[z].any(): ax.contour(WTg[z], levels=[0.5], colors="#2d4dff", linewidths=2.0, linestyles="--")

        ax.set_title(f"z={z} — Pred (solid) vs GT (dashed)")
        ax.axis("off")
        if title: fig.suptitle(title, fontsize=14)
        plt.tight_layout()
        if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()

    print(f"[INFO] Menampilkan slice z={z}")
    return z

def show_overlay_grid(z_indices, cols=3, title=None, save_path=None):
    rows = math.ceil(len(z_indices)/cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
    axes = np.array(axes).reshape(rows, cols)

    # normalisasi window supaya kontras
    lo, hi = np.percentile(flair, 1), np.percentile(flair, 99)

    for i, z in enumerate(z_indices):
        r, c = divmod(i, cols)
        ax = axes[r, c]
        ax.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)

        # kontur tebal
        if ET[z].any():
            ax.contour(ET[z], levels=[0.5], colors='#ff2d2d', linewidths=2.0)
        if TC[z].any():
            ax.contour(TC[z], levels=[0.5], colors='#2dcc2d', linewidths=2.0)
        if WT[z].any():
            ax.contour(WT[z], levels=[0.5], colors='#2d4dff', linewidths=2.0)

        # overlay fill tipis (opsional)
        et_fill = np.ma.masked_where(ET[z]==0, ET[z])
        tc_fill = np.ma.masked_where(TC[z]==0, TC[z])
        wt_fill = np.ma.masked_where(WT[z]==0, WT[z])
        ax.imshow(et_fill, cmap='Reds',   alpha=0.25, vmin=0, vmax=1)
        ax.imshow(tc_fill, cmap='Greens', alpha=0.25, vmin=0, vmax=1)
        ax.imshow(wt_fill, cmap='Blues',  alpha=0.25, vmin=0, vmax=1)

        ax.set_title(f"z={z} | ET red, TC green, WT blue")
        ax.axis("off")

    from matplotlib.patches import Patch
    handles = [Patch(color='#ff2d2d', label='ET'),
               Patch(color='#2dcc2d', label='TC'),
               Patch(color='#2d4dff', label='WT')]
    if title: fig.suptitle(title, fontsize=14)
    fig.legend(handles=handles, loc='upper right')
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

def show_gt_overlay(z_indices, flair, gt, cols=3, title=None, save_path=None):
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch
    import math

    ETg, TCg, WTg = gt[0], gt[1], gt[2]
    rows = math.ceil(len(z_indices)/cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
    axes = np.array(axes).reshape(rows, cols)
    lo, hi = np.percentile(flair, 1), np.percentile(flair, 99)

    for i, z in enumerate(z_indices):
        r, c = divmod(i, cols)
        ax = axes[r, c]
        ax.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        # fill tipis
        ax.imshow(np.ma.masked_where(ETg[z]==0, ETg[z]), cmap='Reds',   alpha=0.25, vmin=0, vmax=1)
        ax.imshow(np.ma.masked_where(TCg[z]==0, TCg[z]), cmap='Greens', alpha=0.25, vmin=0, vmax=1)
        ax.imshow(np.ma.masked_where(WTg[z]==0, WTg[z]), cmap='Blues',  alpha=0.25, vmin=0, vmax=1)
        # kontur putus-putus
        if ETg[z].any(): ax.contour(ETg[z], levels=[0.5], colors='#ff2d2d', linewidths=2, linestyles='--')
        if TCg[z].any(): ax.contour(TCg[z], levels=[0.5], colors='#2dcc2d', linewidths=2, linestyles='--')
        if WTg[z].any(): ax.contour(WTg[z], levels=[0.5], colors='#2d4dff', linewidths=2, linestyles='--')
        ax.set_title(f"z={z} | GT")
        ax.axis("off")

    handles = [Patch(color='#ff2d2d', label='ET (GT)'),
               Patch(color='#2dcc2d', label='TC (GT)'),
               Patch(color='#2d4dff', label='WT (GT)')]
    if title: fig.suptitle(title, fontsize=14)
    fig.legend(handles=handles, loc='upper right')
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


def show_pred_vs_gt(z_indices, flair, pred, gt, title=None, save_path=None):
    """Grid 2 kolom per slice: kiri = Pred, kanan = GT."""
    import numpy as np
    import matplotlib.pyplot as plt
    import math

    ETp, TCp, WTp = pred[0], pred[1], pred[2]
    ETg, TCg, WTg = gt[0],   gt[1],   gt[2]

    rows, cols = len(z_indices), 2
    fig, axes = plt.subplots(rows, cols, figsize=(8*cols, 4*rows))
    lo, hi = np.percentile(flair, 1), np.percentile(flair, 99)

    for i, z in enumerate(z_indices):
        # ---- Pred (kiri)
        axp = axes[i, 0]
        axp.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        axp.imshow(np.ma.masked_where(ETp[z]==0, ETp[z]), cmap='Reds',   alpha=0.25, vmin=0, vmax=1)
        axp.imshow(np.ma.masked_where(TCp[z]==0, TCp[z]), cmap='Greens', alpha=0.25, vmin=0, vmax=1)
        axp.imshow(np.ma.masked_where(WTp[z]==0, WTp[z]), cmap='Blues',  alpha=0.25, vmin=0, vmax=1)
        if ETp[z].any(): axp.contour(ETp[z], levels=[0.5], colors='#ff2d2d', linewidths=2.0)
        if TCp[z].any(): axp.contour(TCp[z], levels=[0.5], colors='#2dcc2d', linewidths=2.0)
        if WTp[z].any(): axp.contour(WTp[z], levels=[0.5], colors='#2d4dff', linewidths=2.0)
        axp.set_title(f"z={z} — Pred")
        axp.axis("off")

        # ---- GT (kanan)
        axg = axes[i, 1]
        axg.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        axg.imshow(np.ma.masked_where(ETg[z]==0, ETg[z]), cmap='Reds',   alpha=0.25, vmin=0, vmax=1)
        axg.imshow(np.ma.masked_where(TCg[z]==0, TCg[z]), cmap='Greens', alpha=0.25, vmin=0, vmax=1)
        axg.imshow(np.ma.masked_where(WTg[z]==0, WTg[z]), cmap='Blues',  alpha=0.25, vmin=0, vmax=1)
        if ETg[z].any(): axg.contour(ETg[z], levels=[0.5], colors='#ff2d2d', linewidths=2, linestyles='--')
        if TCg[z].any(): axg.contour(TCg[z], levels=[0.5], colors='#2dcc2d', linewidths=2, linestyles='--')
        if WTg[z].any(): axg.contour(WTg[z], levels=[0.5], colors='#2d4dff', linewidths=2, linestyles='--')
        axg.set_title(f"z={z} — GT")
        axg.axis("off")

    if title: fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    
def show_pred_vs_gt_pair(flair, pred, gt, z=None, title=None, save_path=None,
                         fill_alpha=0.25, prefer="gt"):
    """
    Tampilkan 1 slice berdampingan: kiri = Prediksi, kanan = Ground Truth (SEG).
    - flair: (D,H,W) float
    - pred : (3,D,H,W) uint8/bool  -> [ET,TC,WT]
    - gt   : (3,D,H,W) uint8/bool  -> [ET,TC,WT]
    - z    : index slice (opsional); jika None dipilih otomatis (prioritas GT)
    """
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch

    # --- validasi bentuk ---
    assert pred.ndim == 4 and pred.shape[0] == 3, "pred harus (3,D,H,W)"
    assert gt.ndim   == 4 and gt.shape[0]   == 3, "gt harus (3,D,H,W)"
    D, H, W = flair.shape
    assert pred.shape[1:] == (D, H, W), "pred & flair harus ukuran sama"
    assert gt.shape[1:]   == (D, H, W), "gt & flair harus ukuran sama"

    # --- pilih slice otomatis bila perlu ---
    def pick_best_slice(_pred, _gt, prefer="gt"):
        cand = None
        if prefer == "gt" and _gt is not None and _gt.any():
            sc = _gt.any(axis=0).sum(axis=(1,2))  # (D,)
            if sc.max() > 0: return int(sc.argmax())
        if _pred is not None and _pred.any():
            sc = _pred.any(axis=0).sum(axis=(1,2))
            if sc.max() > 0: return int(sc.argmax())
        return D // 2

    if z is None:
        z = pick_best_slice(pred, gt, prefer=prefer)

    # --- windowing biar kontras enak ---
    lo, hi = np.percentile(flair, 1), np.percentile(flair, 99)

    ETp, TCp, WTp = pred[0], pred[1], pred[2]
    ETg, TCg, WTg = gt[0],   gt[1],   gt[2]

    fig, (axp, axg) = plt.subplots(1, 2, figsize=(12, 5))

    # ===== KIRI: PREDIKSI =====
    axp.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
    axp.imshow(np.ma.masked_where(ETp[z]==0, ETp[z]), cmap="Reds",   alpha=fill_alpha, vmin=0, vmax=1)
    axp.imshow(np.ma.masked_where(TCp[z]==0, TCp[z]), cmap="Greens", alpha=fill_alpha, vmin=0, vmax=1)
    axp.imshow(np.ma.masked_where(WTp[z]==0, WTp[z]), cmap="Blues",  alpha=fill_alpha, vmin=0, vmax=1)
    if ETp[z].any(): axp.contour(ETp[z], levels=[0.5], colors="#ff2d2d", linewidths=2.0)
    if TCp[z].any(): axp.contour(TCp[z], levels=[0.5], colors="#2dcc2d", linewidths=2.0)
    if WTp[z].any(): axp.contour(WTp[z], levels=[0.5], colors="#2d4dff", linewidths=2.0)
    axp.set_title(f"z={z} — Prediction")
    axp.axis("off")

    # ===== KANAN: GROUND TRUTH =====
    axg.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
    axg.imshow(np.ma.masked_where(ETg[z]==0, ETg[z]), cmap="Reds",   alpha=fill_alpha, vmin=0, vmax=1)
    axg.imshow(np.ma.masked_where(TCg[z]==0, TCg[z]), cmap="Greens", alpha=fill_alpha, vmin=0, vmax=1)
    axg.imshow(np.ma.masked_where(WTg[z]==0, WTg[z]), cmap="Blues",  alpha=fill_alpha, vmin=0, vmax=1)
    if ETg[z].any(): axg.contour(ETg[z], levels=[0.5], colors="#ff2d2d", linewidths=2.0, linestyles="--")
    if TCg[z].any(): axg.contour(TCg[z], levels=[0.5], colors="#2dcc2d", linewidths=2.0, linestyles="--")
    if WTg[z].any(): axg.contour(WTg[z], levels=[0.5], colors="#2d4dff", linewidths=2.0, linestyles="--")
    axg.set_title(f"z={z} — Ground Truth")
    axg.axis("off")

    # legend
    handles = [Patch(color="#ff2d2d", label="ET"),
               Patch(color="#2dcc2d", label="TC"),
               Patch(color="#2d4dff", label="WT")]
    if title: fig.suptitle(title, fontsize=14)
    fig.legend(handles=handles, loc="upper right")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    return z

# Pred-only (sudah ada)
show_overlay_grid(zs, cols=3, title=f"{pid} — {MODEL_TYPE.upper()} (DS Ensemble)")

# Siapkan GT dari sample_t (sudah ter-transform & align dengan img)
gt_t = sample_t["label_ettcwt"]
gt = gt_t.cpu().numpy().astype(np.uint8) if torch.is_tensor(gt_t) else gt_t.astype(np.uint8)

# Pilih slice GT (atau pakai zs supaya slice-nya sama persis dengan pred)
gt_union = gt.any(axis=0)
z_scores_gt = gt_union.sum(axis=(1,2))
zs_gt = np.linspace(0, gt.shape[1]-1, 6, dtype=int) if z_scores_gt.max()==0 else np.sort(np.argsort(-z_scores_gt)[:6])

# GT-only overlay
show_gt_overlay(zs_gt, flair=flair, gt=gt, cols=3, title=f"{pid} — Ground Truth (SEG)")

gt_t = sample_t["label_ettcwt"]
gt   = gt_t.cpu().numpy().astype(np.uint8) if torch.is_tensor(gt_t) else gt_t.astype(np.uint8)

_ = show_pred_vs_gt_pair(
    flair=flair,
    pred=pred,
    gt=gt,
    z=None,  # biarkan auto-pick slice paling informatif
    title=f"{pid} — {MODEL_TYPE.upper()} (Pred vs SEG)",
    save_path=None  # isi path kalau mau disimpan
)

gt_t = sample_t["label_ettcwt"]
gt = gt_t.cpu().numpy().astype(np.uint8) if torch.is_tensor(gt_t) else gt_t.astype(np.uint8)

# Tampilkan satu irisan, overlay Pred vs GT di satu gambar
_ = show_pred_gt_single(flair, pred, gt=gt, z=None, mode="overlay",
                        title=f"{pid} — {MODEL_TYPE.upper()} (1 slice)")


# Predict E3

In [ ]:
# ==== 1) Setup: pilih model & checkpoint ====
import os, math, torch, numpy as np, nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path
from monai.inferers import sliding_window_inference
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_TYPE   = "unet"   # "unetpp" atau "unet"
CKPT_PATH = configured_path("BTS_UNET_LR_CHECKPOINT", CHECKPOINT_ROOT / "unet_warmup_lr" / "unetpp_medicalnet_multilabel_best_20251013-024333.pt")
THRESH       = 0.5        # sigmoid threshold untuk ET/TC/WT
ROI_DEFAULT  = (128,128,128)  # fallback roi (akan di-cap by volume size)

if MODEL_TYPE == "unetpp":
    model = UNetPP3D_MedicalNet(
        in_channels=4, out_channels=3,
        features=(16,32,64,128,128,16),
        enc_channels=(64,64,128,256),
    ).to(device)
elif MODEL_TYPE == "unet":
    # DS ON agar cocok dengan checkpoint training-mu
    model = UNet3D_MedicalNet(
        in_channels=4, out_channels=3,
        features=(16,32,64,128,128),
        enc_channels=(64,64,128,256),
        deep_supervision=True,   # <--- ON (penting)
    ).to(device)
else:
    raise ValueError("MODEL_TYPE harus 'unetpp' atau 'unet'.")

state = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(state, strict=True)
model.eval()
for m in model.modules():
    if isinstance(m, (torch.nn.BatchNorm3d, torch.nn.SyncBatchNorm)):
        m.eval()

# ==== 6) Predictor ENSAMBLE (Opsi B) ====
# Bekerja untuk:
# - UNet biasa DS ON → out = [y_full, y_/2, y_/4, y_/8] → upsample & rata-rata
# - UNet++ eval → out tensor tunggal → langsung return
def predictor_ensemble(x, weights=(1.0, 0.5, 0.25, 0.125)):
    outs = model(x)
    if not isinstance(outs, (list, tuple)):
        return outs  # mis. UNet++ eval mengembalikan single head
    # asumsi urutan: [y_full(/1), y_ds2(/2), y_ds3(/4), y_ds4(/8)]
    y0, y2, y3, y4 = outs
    tgt = y0.shape[2:]  # (D,H,W)
    y2u = F.interpolate(y2, size=tgt, mode="trilinear", align_corners=False)
    y3u = F.interpolate(y3, size=tgt, mode="trilinear", align_corners=False)
    y4u = F.interpolate(y4, size=tgt, mode="trilinear", align_corners=False)
    w = torch.tensor(weights, dtype=y0.dtype, device=y0.device)
    if w.numel() != 4:
        raise ValueError("weights harus berisi 4 angka untuk [y0, y/2, y/4, y/8].")
    w = w / w.sum()
    return w[0]*y0 + w[1]*y2u + w[2]*y3u + w[3]*y4u

# ==== 7) Ambil satu kasus & apply transforms ====
idx = 0
sample_raw = all_paths[idx]
sample_t   = test_transforms(sample_raw)
img = sample_t["image"].unsqueeze(0).to(device)  # [1,4,D,H,W]
pid = sample_t["pid"]

# ==== 8) Sliding-window inference (pakai predictor_ensemble) ====
_, _, D, H, W = img.shape
roi = (min(ROI_DEFAULT[0], D), min(ROI_DEFAULT[1], H), min(ROI_DEFAULT[2], W))
with torch.no_grad():
    logits = sliding_window_inference(
        inputs=img, roi_size=roi, sw_batch_size=1,
        predictor=predictor_ensemble, overlap=0.5, mode="gaussian"
    )  # [1,3,D,H,W]

probs = torch.sigmoid(logits)[0].cpu().numpy()  # (3,D,H,W)
pred  = (probs > THRESH).astype(np.uint8)

# --- sanity check ---
print("prob stats: min/max/mean =", float(probs.min()), float(probs.max()), float(probs.mean()))
voxel_counts = pred.reshape(3, -1).sum(1)
print("voxel counts  ET/TC/WT  =", voxel_counts.tolist())

# per-slice (axial) jumlah voxel bertopeng
per_slice = pred.any(axis=0).sum(axis=(1,2))
print("top slices by prediction:", np.argsort(-per_slice)[:10])


# ==== 9) Pilih irisan informatif ====
tumor_map = pred.any(axis=0)            # (D,H,W)
z_scores  = tumor_map.sum(axis=(1,2))   # (D,)
if z_scores.max() == 0:
    zs = np.linspace(0, D-1, 6, dtype=int)
else:
    zs = np.sort(np.argsort(-z_scores)[:6])

# ==== 10) Visualisasi overlay (FLAIR) ====
flair = img[0,0].cpu().numpy()
ET, TC, WT = pred[0], pred[1], pred[2]



def pick_best_slice(flair, pred=None, gt=None, prefer="gt"):
    """
    Pilih irisan (z) yang paling 'ramai' tumor.
    Urutan preferensi: GT -> Pred -> tengah volume.
    """
    z = None
    if prefer == "gt" and gt is not None:
        uni = gt.any(axis=0)                      # (D,H,W)
        sc  = uni.sum(axis=(1,2))                 # (D,)
        if sc.max() > 0:
            z = int(sc.argmax())
    if z is None and pred is not None:
        uni = pred.any(axis=0)
        sc  = uni.sum(axis=(1,2))
        if sc.max() > 0:
            z = int(sc.argmax())
    if z is None:
        z = flair.shape[0] // 2
    return z

def show_pred_gt_single(flair, pred, gt=None, z=None, mode="overlay", title=None, save_path=None):
    """
    mode='overlay'  -> 1 gambar, Pred (solid) + GT (putus-putus) di axis yang sama.
    mode='split'    -> 2 gambar berdampingan: kiri=Pred, kanan=GT.
    """
    # Siapkan slice index
    if z is None:
        z = pick_best_slice(flair, pred=pred, gt=gt, prefer="gt")  # pilih otomatis

    lo, hi = np.percentile(flair, 1), np.percentile(flair, 99)
    ETp, TCp, WTp = pred[0], pred[1], pred[2]
    ETg = TCg = WTg = None
    if gt is not None:
        ETg, TCg, WTg = gt[0], gt[1], gt[2]

    if mode == "split":
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axp, axg = axes

        # ---- PRED (kiri)
        axp.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        axp.imshow(np.ma.masked_where(ETp[z]==0, ETp[z]), cmap="Reds",   alpha=0.25, vmin=0, vmax=1)
        axp.imshow(np.ma.masked_where(TCp[z]==0, TCp[z]), cmap="Greens", alpha=0.25, vmin=0, vmax=1)
        axp.imshow(np.ma.masked_where(WTp[z]==0, WTp[z]), cmap="Blues",  alpha=0.25, vmin=0, vmax=1)
        if ETp[z].any(): axp.contour(ETp[z], levels=[0.5], colors="#ff2d2d", linewidths=2.0)
        if TCp[z].any(): axp.contour(TCp[z], levels=[0.5], colors="#2dcc2d", linewidths=2.0)
        if WTp[z].any(): axp.contour(WTp[z], levels=[0.5], colors="#2d4dff", linewidths=2.0)
        axp.set_title(f"z={z} — Prediction")
        axp.axis("off")

        # ---- GT (kanan)
        axg.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        if gt is not None:
            axg.imshow(np.ma.masked_where(ETg[z]==0, ETg[z]), cmap="Reds",   alpha=0.25, vmin=0, vmax=1)
            axg.imshow(np.ma.masked_where(TCg[z]==0, TCg[z]), cmap="Greens", alpha=0.25, vmin=0, vmax=1)
            axg.imshow(np.ma.masked_where(WTg[z]==0, WTg[z]), cmap="Blues",  alpha=0.25, vmin=0, vmax=1)
            if ETg[z].any(): axg.contour(ETg[z], levels=[0.5], colors="#ff2d2d", linewidths=2, linestyles="--")
            if TCg[z].any(): axg.contour(TCg[z], levels=[0.5], colors="#2dcc2d", linewidths=2, linestyles="--")
            if WTg[z].any(): axg.contour(WTg[z], levels=[0.5], colors="#2d4dff", linewidths=2, linestyles="--")
        axg.set_title(f"z={z} — Ground Truth")
        axg.axis("off")

        if title: fig.suptitle(title, fontsize=14)
        plt.tight_layout()
        if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()

    else:  # mode == "overlay"
        fig, ax = plt.subplots(1, 1, figsize=(6, 6))
        ax.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        # Pred (solid)
        if ETp[z].any(): ax.contour(ETp[z], levels=[0.5], colors="#ff2d2d", linewidths=2.0)
        if TCp[z].any(): ax.contour(TCp[z], levels=[0.5], colors="#2dcc2d", linewidths=2.0)
        if WTp[z].any(): ax.contour(WTp[z], levels=[0.5], colors="#2d4dff", linewidths=2.0)
        # GT (dashed)
        if gt is not None:
            if ETg[z].any(): ax.contour(ETg[z], levels=[0.5], colors="#ff2d2d", linewidths=2.0, linestyles="--")
            if TCg[z].any(): ax.contour(TCg[z], levels=[0.5], colors="#2dcc2d", linewidths=2.0, linestyles="--")
            if WTg[z].any(): ax.contour(WTg[z], levels=[0.5], colors="#2d4dff", linewidths=2.0, linestyles="--")

        ax.set_title(f"z={z} — Pred (solid) vs GT (dashed)")
        ax.axis("off")
        if title: fig.suptitle(title, fontsize=14)
        plt.tight_layout()
        if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()

    print(f"[INFO] Menampilkan slice z={z}")
    return z

def show_overlay_grid(z_indices, cols=3, title=None, save_path=None):
    rows = math.ceil(len(z_indices)/cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
    axes = np.array(axes).reshape(rows, cols)

    # normalisasi window supaya kontras
    lo, hi = np.percentile(flair, 1), np.percentile(flair, 99)

    for i, z in enumerate(z_indices):
        r, c = divmod(i, cols)
        ax = axes[r, c]
        ax.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)

        # kontur tebal
        if ET[z].any():
            ax.contour(ET[z], levels=[0.5], colors='#ff2d2d', linewidths=2.0)
        if TC[z].any():
            ax.contour(TC[z], levels=[0.5], colors='#2dcc2d', linewidths=2.0)
        if WT[z].any():
            ax.contour(WT[z], levels=[0.5], colors='#2d4dff', linewidths=2.0)

        # overlay fill tipis (opsional)
        et_fill = np.ma.masked_where(ET[z]==0, ET[z])
        tc_fill = np.ma.masked_where(TC[z]==0, TC[z])
        wt_fill = np.ma.masked_where(WT[z]==0, WT[z])
        ax.imshow(et_fill, cmap='Reds',   alpha=0.25, vmin=0, vmax=1)
        ax.imshow(tc_fill, cmap='Greens', alpha=0.25, vmin=0, vmax=1)
        ax.imshow(wt_fill, cmap='Blues',  alpha=0.25, vmin=0, vmax=1)

        ax.set_title(f"z={z} | ET red, TC green, WT blue")
        ax.axis("off")

    from matplotlib.patches import Patch
    handles = [Patch(color='#ff2d2d', label='ET'),
               Patch(color='#2dcc2d', label='TC'),
               Patch(color='#2d4dff', label='WT')]
    if title: fig.suptitle(title, fontsize=14)
    fig.legend(handles=handles, loc='upper right')
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

def show_gt_overlay(z_indices, flair, gt, cols=3, title=None, save_path=None):
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch
    import math

    ETg, TCg, WTg = gt[0], gt[1], gt[2]
    rows = math.ceil(len(z_indices)/cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
    axes = np.array(axes).reshape(rows, cols)
    lo, hi = np.percentile(flair, 1), np.percentile(flair, 99)

    for i, z in enumerate(z_indices):
        r, c = divmod(i, cols)
        ax = axes[r, c]
        ax.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        # fill tipis
        ax.imshow(np.ma.masked_where(ETg[z]==0, ETg[z]), cmap='Reds',   alpha=0.25, vmin=0, vmax=1)
        ax.imshow(np.ma.masked_where(TCg[z]==0, TCg[z]), cmap='Greens', alpha=0.25, vmin=0, vmax=1)
        ax.imshow(np.ma.masked_where(WTg[z]==0, WTg[z]), cmap='Blues',  alpha=0.25, vmin=0, vmax=1)
        # kontur putus-putus
        if ETg[z].any(): ax.contour(ETg[z], levels=[0.5], colors='#ff2d2d', linewidths=2, linestyles='--')
        if TCg[z].any(): ax.contour(TCg[z], levels=[0.5], colors='#2dcc2d', linewidths=2, linestyles='--')
        if WTg[z].any(): ax.contour(WTg[z], levels=[0.5], colors='#2d4dff', linewidths=2, linestyles='--')
        ax.set_title(f"z={z} | GT")
        ax.axis("off")

    handles = [Patch(color='#ff2d2d', label='ET (GT)'),
               Patch(color='#2dcc2d', label='TC (GT)'),
               Patch(color='#2d4dff', label='WT (GT)')]
    if title: fig.suptitle(title, fontsize=14)
    fig.legend(handles=handles, loc='upper right')
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


def show_pred_vs_gt(z_indices, flair, pred, gt, title=None, save_path=None):
    """Grid 2 kolom per slice: kiri = Pred, kanan = GT."""
    import numpy as np
    import matplotlib.pyplot as plt
    import math

    ETp, TCp, WTp = pred[0], pred[1], pred[2]
    ETg, TCg, WTg = gt[0],   gt[1],   gt[2]

    rows, cols = len(z_indices), 2
    fig, axes = plt.subplots(rows, cols, figsize=(8*cols, 4*rows))
    lo, hi = np.percentile(flair, 1), np.percentile(flair, 99)

    for i, z in enumerate(z_indices):
        # ---- Pred (kiri)
        axp = axes[i, 0]
        axp.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        axp.imshow(np.ma.masked_where(ETp[z]==0, ETp[z]), cmap='Reds',   alpha=0.25, vmin=0, vmax=1)
        axp.imshow(np.ma.masked_where(TCp[z]==0, TCp[z]), cmap='Greens', alpha=0.25, vmin=0, vmax=1)
        axp.imshow(np.ma.masked_where(WTp[z]==0, WTp[z]), cmap='Blues',  alpha=0.25, vmin=0, vmax=1)
        if ETp[z].any(): axp.contour(ETp[z], levels=[0.5], colors='#ff2d2d', linewidths=2.0)
        if TCp[z].any(): axp.contour(TCp[z], levels=[0.5], colors='#2dcc2d', linewidths=2.0)
        if WTp[z].any(): axp.contour(WTp[z], levels=[0.5], colors='#2d4dff', linewidths=2.0)
        axp.set_title(f"z={z} — Pred")
        axp.axis("off")

        # ---- GT (kanan)
        axg = axes[i, 1]
        axg.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        axg.imshow(np.ma.masked_where(ETg[z]==0, ETg[z]), cmap='Reds',   alpha=0.25, vmin=0, vmax=1)
        axg.imshow(np.ma.masked_where(TCg[z]==0, TCg[z]), cmap='Greens', alpha=0.25, vmin=0, vmax=1)
        axg.imshow(np.ma.masked_where(WTg[z]==0, WTg[z]), cmap='Blues',  alpha=0.25, vmin=0, vmax=1)
        if ETg[z].any(): axg.contour(ETg[z], levels=[0.5], colors='#ff2d2d', linewidths=2, linestyles='--')
        if TCg[z].any(): axg.contour(TCg[z], levels=[0.5], colors='#2dcc2d', linewidths=2, linestyles='--')
        if WTg[z].any(): axg.contour(WTg[z], levels=[0.5], colors='#2d4dff', linewidths=2, linestyles='--')
        axg.set_title(f"z={z} — GT")
        axg.axis("off")

    if title: fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    
def show_pred_vs_gt_pair(flair, pred, gt, z=None, title=None, save_path=None,
                         fill_alpha=0.25, prefer="gt"):
    """
    Tampilkan 1 slice berdampingan: kiri = Prediksi, kanan = Ground Truth (SEG).
    - flair: (D,H,W) float
    - pred : (3,D,H,W) uint8/bool  -> [ET,TC,WT]
    - gt   : (3,D,H,W) uint8/bool  -> [ET,TC,WT]
    - z    : index slice (opsional); jika None dipilih otomatis (prioritas GT)
    """
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch

    # --- validasi bentuk ---
    assert pred.ndim == 4 and pred.shape[0] == 3, "pred harus (3,D,H,W)"
    assert gt.ndim   == 4 and gt.shape[0]   == 3, "gt harus (3,D,H,W)"
    D, H, W = flair.shape
    assert pred.shape[1:] == (D, H, W), "pred & flair harus ukuran sama"
    assert gt.shape[1:]   == (D, H, W), "gt & flair harus ukuran sama"

    # --- pilih slice otomatis bila perlu ---
    def pick_best_slice(_pred, _gt, prefer="gt"):
        cand = None
        if prefer == "gt" and _gt is not None and _gt.any():
            sc = _gt.any(axis=0).sum(axis=(1,2))  # (D,)
            if sc.max() > 0: return int(sc.argmax())
        if _pred is not None and _pred.any():
            sc = _pred.any(axis=0).sum(axis=(1,2))
            if sc.max() > 0: return int(sc.argmax())
        return D // 2

    if z is None:
        z = pick_best_slice(pred, gt, prefer=prefer)

    # --- windowing biar kontras enak ---
    lo, hi = np.percentile(flair, 1), np.percentile(flair, 99)

    ETp, TCp, WTp = pred[0], pred[1], pred[2]
    ETg, TCg, WTg = gt[0],   gt[1],   gt[2]

    fig, (axp, axg) = plt.subplots(1, 2, figsize=(12, 5))

    # ===== KIRI: PREDIKSI =====
    axp.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
    axp.imshow(np.ma.masked_where(ETp[z]==0, ETp[z]), cmap="Reds",   alpha=fill_alpha, vmin=0, vmax=1)
    axp.imshow(np.ma.masked_where(TCp[z]==0, TCp[z]), cmap="Greens", alpha=fill_alpha, vmin=0, vmax=1)
    axp.imshow(np.ma.masked_where(WTp[z]==0, WTp[z]), cmap="Blues",  alpha=fill_alpha, vmin=0, vmax=1)
    if ETp[z].any(): axp.contour(ETp[z], levels=[0.5], colors="#ff2d2d", linewidths=2.0)
    if TCp[z].any(): axp.contour(TCp[z], levels=[0.5], colors="#2dcc2d", linewidths=2.0)
    if WTp[z].any(): axp.contour(WTp[z], levels=[0.5], colors="#2d4dff", linewidths=2.0)
    axp.set_title(f"z={z} — Prediction")
    axp.axis("off")

    # ===== KANAN: GROUND TRUTH =====
    axg.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
    axg.imshow(np.ma.masked_where(ETg[z]==0, ETg[z]), cmap="Reds",   alpha=fill_alpha, vmin=0, vmax=1)
    axg.imshow(np.ma.masked_where(TCg[z]==0, TCg[z]), cmap="Greens", alpha=fill_alpha, vmin=0, vmax=1)
    axg.imshow(np.ma.masked_where(WTg[z]==0, WTg[z]), cmap="Blues",  alpha=fill_alpha, vmin=0, vmax=1)
    if ETg[z].any(): axg.contour(ETg[z], levels=[0.5], colors="#ff2d2d", linewidths=2.0, linestyles="--")
    if TCg[z].any(): axg.contour(TCg[z], levels=[0.5], colors="#2dcc2d", linewidths=2.0, linestyles="--")
    if WTg[z].any(): axg.contour(WTg[z], levels=[0.5], colors="#2d4dff", linewidths=2.0, linestyles="--")
    axg.set_title(f"z={z} — Ground Truth")
    axg.axis("off")

    # legend
    handles = [Patch(color="#ff2d2d", label="ET"),
               Patch(color="#2dcc2d", label="TC"),
               Patch(color="#2d4dff", label="WT")]
    if title: fig.suptitle(title, fontsize=14)
    fig.legend(handles=handles, loc="upper right")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    return z

# Pred-only (sudah ada)
show_overlay_grid(zs, cols=3, title=f"{pid} — {MODEL_TYPE.upper()} (DS Ensemble)")

# Siapkan GT dari sample_t (sudah ter-transform & align dengan img)
gt_t = sample_t["label_ettcwt"]
gt = gt_t.cpu().numpy().astype(np.uint8) if torch.is_tensor(gt_t) else gt_t.astype(np.uint8)

# Pilih slice GT (atau pakai zs supaya slice-nya sama persis dengan pred)
gt_union = gt.any(axis=0)
z_scores_gt = gt_union.sum(axis=(1,2))
zs_gt = np.linspace(0, gt.shape[1]-1, 6, dtype=int) if z_scores_gt.max()==0 else np.sort(np.argsort(-z_scores_gt)[:6])

# GT-only overlay
show_gt_overlay(zs_gt, flair=flair, gt=gt, cols=3, title=f"{pid} — Ground Truth (SEG)")

gt_t = sample_t["label_ettcwt"]
gt   = gt_t.cpu().numpy().astype(np.uint8) if torch.is_tensor(gt_t) else gt_t.astype(np.uint8)

_ = show_pred_vs_gt_pair(
    flair=flair,
    pred=pred,
    gt=gt,
    z=None,  # biarkan auto-pick slice paling informatif
    title=f"{pid} — {MODEL_TYPE.upper()} (Pred vs SEG)",
    save_path=None  # isi path kalau mau disimpan
)

gt_t = sample_t["label_ettcwt"]
gt = gt_t.cpu().numpy().astype(np.uint8) if torch.is_tensor(gt_t) else gt_t.astype(np.uint8)

# Tampilkan satu irisan, overlay Pred vs GT di satu gambar
_ = show_pred_gt_single(flair, pred, gt=gt, z=None, mode="overlay",
                        title=f"{pid} — {MODEL_TYPE.upper()} (1 slice)")


# Prediksi E4

In [ ]:
# ==== 1) Setup: pilih model & checkpoint ====
import os, math, torch, numpy as np, nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path
from monai.inferers import sliding_window_inference
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_TYPE   = "unet"   # "unetpp" atau "unet"
CKPT_PATH = configured_path("BTS_UNET_FREEZE_CHECKPOINT", CHECKPOINT_ROOT / "unet_warmup_freeze" / "unetpp_medicalnet_multilabel_best_20251014-030104.pt")
THRESH       = 0.5        # sigmoid threshold untuk ET/TC/WT
ROI_DEFAULT  = (128,128,128)  # fallback roi (akan di-cap by volume size)

if MODEL_TYPE == "unetpp":
    model = UNetPP3D_MedicalNet(
        in_channels=4, out_channels=3,
        features=(16,32,64,128,128,16),
        enc_channels=(64,64,128,256),
    ).to(device)
elif MODEL_TYPE == "unet":
    # DS ON agar cocok dengan checkpoint training-mu
    model = UNet3D_MedicalNet(
        in_channels=4, out_channels=3,
        features=(16,32,64,128,128),
        enc_channels=(64,64,128,256),
        deep_supervision=True,   # <--- ON (penting)
    ).to(device)
else:
    raise ValueError("MODEL_TYPE harus 'unetpp' atau 'unet'.")

state = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(state, strict=True)
model.eval()
for m in model.modules():
    if isinstance(m, (torch.nn.BatchNorm3d, torch.nn.SyncBatchNorm)):
        m.eval()

# ==== 6) Predictor ENSAMBLE (Opsi B) ====
# Bekerja untuk:
# - UNet biasa DS ON → out = [y_full, y_/2, y_/4, y_/8] → upsample & rata-rata
# - UNet++ eval → out tensor tunggal → langsung return
def predictor_ensemble(x, weights=(1.0, 0.5, 0.25, 0.125)):
    outs = model(x)
    if not isinstance(outs, (list, tuple)):
        return outs  # mis. UNet++ eval mengembalikan single head
    # asumsi urutan: [y_full(/1), y_ds2(/2), y_ds3(/4), y_ds4(/8)]
    y0, y2, y3, y4 = outs
    tgt = y0.shape[2:]  # (D,H,W)
    y2u = F.interpolate(y2, size=tgt, mode="trilinear", align_corners=False)
    y3u = F.interpolate(y3, size=tgt, mode="trilinear", align_corners=False)
    y4u = F.interpolate(y4, size=tgt, mode="trilinear", align_corners=False)
    w = torch.tensor(weights, dtype=y0.dtype, device=y0.device)
    if w.numel() != 4:
        raise ValueError("weights harus berisi 4 angka untuk [y0, y/2, y/4, y/8].")
    w = w / w.sum()
    return w[0]*y0 + w[1]*y2u + w[2]*y3u + w[3]*y4u

# ==== 7) Ambil satu kasus & apply transforms ====
idx = 0
sample_raw = all_paths[idx]
sample_t   = test_transforms(sample_raw)
img = sample_t["image"].unsqueeze(0).to(device)  # [1,4,D,H,W]
pid = sample_t["pid"]

# ==== 8) Sliding-window inference (pakai predictor_ensemble) ====
_, _, D, H, W = img.shape
roi = (min(ROI_DEFAULT[0], D), min(ROI_DEFAULT[1], H), min(ROI_DEFAULT[2], W))
with torch.no_grad():
    logits = sliding_window_inference(
        inputs=img, roi_size=roi, sw_batch_size=1,
        predictor=predictor_ensemble, overlap=0.5, mode="gaussian"
    )  # [1,3,D,H,W]

probs = torch.sigmoid(logits)[0].cpu().numpy()  # (3,D,H,W)
pred  = (probs > THRESH).astype(np.uint8)

# --- sanity check ---
print("prob stats: min/max/mean =", float(probs.min()), float(probs.max()), float(probs.mean()))
voxel_counts = pred.reshape(3, -1).sum(1)
print("voxel counts  ET/TC/WT  =", voxel_counts.tolist())

# per-slice (axial) jumlah voxel bertopeng
per_slice = pred.any(axis=0).sum(axis=(1,2))
print("top slices by prediction:", np.argsort(-per_slice)[:10])


# ==== 9) Pilih irisan informatif ====
tumor_map = pred.any(axis=0)            # (D,H,W)
z_scores  = tumor_map.sum(axis=(1,2))   # (D,)
if z_scores.max() == 0:
    zs = np.linspace(0, D-1, 6, dtype=int)
else:
    zs = np.sort(np.argsort(-z_scores)[:6])

# ==== 10) Visualisasi overlay (FLAIR) ====
flair = img[0,0].cpu().numpy()
ET, TC, WT = pred[0], pred[1], pred[2]



def pick_best_slice(flair, pred=None, gt=None, prefer="gt"):
    """
    Pilih irisan (z) yang paling 'ramai' tumor.
    Urutan preferensi: GT -> Pred -> tengah volume.
    """
    z = None
    if prefer == "gt" and gt is not None:
        uni = gt.any(axis=0)                      # (D,H,W)
        sc  = uni.sum(axis=(1,2))                 # (D,)
        if sc.max() > 0:
            z = int(sc.argmax())
    if z is None and pred is not None:
        uni = pred.any(axis=0)
        sc  = uni.sum(axis=(1,2))
        if sc.max() > 0:
            z = int(sc.argmax())
    if z is None:
        z = flair.shape[0] // 2
    return z

def show_pred_gt_single(flair, pred, gt=None, z=None, mode="overlay", title=None, save_path=None):
    """
    mode='overlay'  -> 1 gambar, Pred (solid) + GT (putus-putus) di axis yang sama.
    mode='split'    -> 2 gambar berdampingan: kiri=Pred, kanan=GT.
    """
    # Siapkan slice index
    if z is None:
        z = pick_best_slice(flair, pred=pred, gt=gt, prefer="gt")  # pilih otomatis

    lo, hi = np.percentile(flair, 1), np.percentile(flair, 99)
    ETp, TCp, WTp = pred[0], pred[1], pred[2]
    ETg = TCg = WTg = None
    if gt is not None:
        ETg, TCg, WTg = gt[0], gt[1], gt[2]

    if mode == "split":
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axp, axg = axes

        # ---- PRED (kiri)
        axp.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        axp.imshow(np.ma.masked_where(ETp[z]==0, ETp[z]), cmap="Reds",   alpha=0.25, vmin=0, vmax=1)
        axp.imshow(np.ma.masked_where(TCp[z]==0, TCp[z]), cmap="Greens", alpha=0.25, vmin=0, vmax=1)
        axp.imshow(np.ma.masked_where(WTp[z]==0, WTp[z]), cmap="Blues",  alpha=0.25, vmin=0, vmax=1)
        if ETp[z].any(): axp.contour(ETp[z], levels=[0.5], colors="#ff2d2d", linewidths=2.0)
        if TCp[z].any(): axp.contour(TCp[z], levels=[0.5], colors="#2dcc2d", linewidths=2.0)
        if WTp[z].any(): axp.contour(WTp[z], levels=[0.5], colors="#2d4dff", linewidths=2.0)
        axp.set_title(f"z={z} — Prediction")
        axp.axis("off")

        # ---- GT (kanan)
        axg.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        if gt is not None:
            axg.imshow(np.ma.masked_where(ETg[z]==0, ETg[z]), cmap="Reds",   alpha=0.25, vmin=0, vmax=1)
            axg.imshow(np.ma.masked_where(TCg[z]==0, TCg[z]), cmap="Greens", alpha=0.25, vmin=0, vmax=1)
            axg.imshow(np.ma.masked_where(WTg[z]==0, WTg[z]), cmap="Blues",  alpha=0.25, vmin=0, vmax=1)
            if ETg[z].any(): axg.contour(ETg[z], levels=[0.5], colors="#ff2d2d", linewidths=2, linestyles="--")
            if TCg[z].any(): axg.contour(TCg[z], levels=[0.5], colors="#2dcc2d", linewidths=2, linestyles="--")
            if WTg[z].any(): axg.contour(WTg[z], levels=[0.5], colors="#2d4dff", linewidths=2, linestyles="--")
        axg.set_title(f"z={z} — Ground Truth")
        axg.axis("off")

        if title: fig.suptitle(title, fontsize=14)
        plt.tight_layout()
        if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()

    else:  # mode == "overlay"
        fig, ax = plt.subplots(1, 1, figsize=(6, 6))
        ax.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        # Pred (solid)
        if ETp[z].any(): ax.contour(ETp[z], levels=[0.5], colors="#ff2d2d", linewidths=2.0)
        if TCp[z].any(): ax.contour(TCp[z], levels=[0.5], colors="#2dcc2d", linewidths=2.0)
        if WTp[z].any(): ax.contour(WTp[z], levels=[0.5], colors="#2d4dff", linewidths=2.0)
        # GT (dashed)
        if gt is not None:
            if ETg[z].any(): ax.contour(ETg[z], levels=[0.5], colors="#ff2d2d", linewidths=2.0, linestyles="--")
            if TCg[z].any(): ax.contour(TCg[z], levels=[0.5], colors="#2dcc2d", linewidths=2.0, linestyles="--")
            if WTg[z].any(): ax.contour(WTg[z], levels=[0.5], colors="#2d4dff", linewidths=2.0, linestyles="--")

        ax.set_title(f"z={z} — Pred (solid) vs GT (dashed)")
        ax.axis("off")
        if title: fig.suptitle(title, fontsize=14)
        plt.tight_layout()
        if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()

    print(f"[INFO] Menampilkan slice z={z}")
    return z

def show_overlay_grid(z_indices, cols=3, title=None, save_path=None):
    rows = math.ceil(len(z_indices)/cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
    axes = np.array(axes).reshape(rows, cols)

    # normalisasi window supaya kontras
    lo, hi = np.percentile(flair, 1), np.percentile(flair, 99)

    for i, z in enumerate(z_indices):
        r, c = divmod(i, cols)
        ax = axes[r, c]
        ax.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)

        # kontur tebal
        if ET[z].any():
            ax.contour(ET[z], levels=[0.5], colors='#ff2d2d', linewidths=2.0)
        if TC[z].any():
            ax.contour(TC[z], levels=[0.5], colors='#2dcc2d', linewidths=2.0)
        if WT[z].any():
            ax.contour(WT[z], levels=[0.5], colors='#2d4dff', linewidths=2.0)

        # overlay fill tipis (opsional)
        et_fill = np.ma.masked_where(ET[z]==0, ET[z])
        tc_fill = np.ma.masked_where(TC[z]==0, TC[z])
        wt_fill = np.ma.masked_where(WT[z]==0, WT[z])
        ax.imshow(et_fill, cmap='Reds',   alpha=0.25, vmin=0, vmax=1)
        ax.imshow(tc_fill, cmap='Greens', alpha=0.25, vmin=0, vmax=1)
        ax.imshow(wt_fill, cmap='Blues',  alpha=0.25, vmin=0, vmax=1)

        ax.set_title(f"z={z} | ET red, TC green, WT blue")
        ax.axis("off")

    from matplotlib.patches import Patch
    handles = [Patch(color='#ff2d2d', label='ET'),
               Patch(color='#2dcc2d', label='TC'),
               Patch(color='#2d4dff', label='WT')]
    if title: fig.suptitle(title, fontsize=14)
    fig.legend(handles=handles, loc='upper right')
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

def show_gt_overlay(z_indices, flair, gt, cols=3, title=None, save_path=None):
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch
    import math

    ETg, TCg, WTg = gt[0], gt[1], gt[2]
    rows = math.ceil(len(z_indices)/cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
    axes = np.array(axes).reshape(rows, cols)
    lo, hi = np.percentile(flair, 1), np.percentile(flair, 99)

    for i, z in enumerate(z_indices):
        r, c = divmod(i, cols)
        ax = axes[r, c]
        ax.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        # fill tipis
        ax.imshow(np.ma.masked_where(ETg[z]==0, ETg[z]), cmap='Reds',   alpha=0.25, vmin=0, vmax=1)
        ax.imshow(np.ma.masked_where(TCg[z]==0, TCg[z]), cmap='Greens', alpha=0.25, vmin=0, vmax=1)
        ax.imshow(np.ma.masked_where(WTg[z]==0, WTg[z]), cmap='Blues',  alpha=0.25, vmin=0, vmax=1)
        # kontur putus-putus
        if ETg[z].any(): ax.contour(ETg[z], levels=[0.5], colors='#ff2d2d', linewidths=2, linestyles='--')
        if TCg[z].any(): ax.contour(TCg[z], levels=[0.5], colors='#2dcc2d', linewidths=2, linestyles='--')
        if WTg[z].any(): ax.contour(WTg[z], levels=[0.5], colors='#2d4dff', linewidths=2, linestyles='--')
        ax.set_title(f"z={z} | GT")
        ax.axis("off")

    handles = [Patch(color='#ff2d2d', label='ET (GT)'),
               Patch(color='#2dcc2d', label='TC (GT)'),
               Patch(color='#2d4dff', label='WT (GT)')]
    if title: fig.suptitle(title, fontsize=14)
    fig.legend(handles=handles, loc='upper right')
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


def show_pred_vs_gt(z_indices, flair, pred, gt, title=None, save_path=None):
    """Grid 2 kolom per slice: kiri = Pred, kanan = GT."""
    import numpy as np
    import matplotlib.pyplot as plt
    import math

    ETp, TCp, WTp = pred[0], pred[1], pred[2]
    ETg, TCg, WTg = gt[0],   gt[1],   gt[2]

    rows, cols = len(z_indices), 2
    fig, axes = plt.subplots(rows, cols, figsize=(8*cols, 4*rows))
    lo, hi = np.percentile(flair, 1), np.percentile(flair, 99)

    for i, z in enumerate(z_indices):
        # ---- Pred (kiri)
        axp = axes[i, 0]
        axp.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        axp.imshow(np.ma.masked_where(ETp[z]==0, ETp[z]), cmap='Reds',   alpha=0.25, vmin=0, vmax=1)
        axp.imshow(np.ma.masked_where(TCp[z]==0, TCp[z]), cmap='Greens', alpha=0.25, vmin=0, vmax=1)
        axp.imshow(np.ma.masked_where(WTp[z]==0, WTp[z]), cmap='Blues',  alpha=0.25, vmin=0, vmax=1)
        if ETp[z].any(): axp.contour(ETp[z], levels=[0.5], colors='#ff2d2d', linewidths=2.0)
        if TCp[z].any(): axp.contour(TCp[z], levels=[0.5], colors='#2dcc2d', linewidths=2.0)
        if WTp[z].any(): axp.contour(WTp[z], levels=[0.5], colors='#2d4dff', linewidths=2.0)
        axp.set_title(f"z={z} — Pred")
        axp.axis("off")

        # ---- GT (kanan)
        axg = axes[i, 1]
        axg.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        axg.imshow(np.ma.masked_where(ETg[z]==0, ETg[z]), cmap='Reds',   alpha=0.25, vmin=0, vmax=1)
        axg.imshow(np.ma.masked_where(TCg[z]==0, TCg[z]), cmap='Greens', alpha=0.25, vmin=0, vmax=1)
        axg.imshow(np.ma.masked_where(WTg[z]==0, WTg[z]), cmap='Blues',  alpha=0.25, vmin=0, vmax=1)
        if ETg[z].any(): axg.contour(ETg[z], levels=[0.5], colors='#ff2d2d', linewidths=2, linestyles='--')
        if TCg[z].any(): axg.contour(TCg[z], levels=[0.5], colors='#2dcc2d', linewidths=2, linestyles='--')
        if WTg[z].any(): axg.contour(WTg[z], levels=[0.5], colors='#2d4dff', linewidths=2, linestyles='--')
        axg.set_title(f"z={z} — GT")
        axg.axis("off")

    if title: fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    
def show_pred_vs_gt_pair(flair, pred, gt, z=None, title=None, save_path=None,
                         fill_alpha=0.25, prefer="gt"):
    """
    Tampilkan 1 slice berdampingan: kiri = Prediksi, kanan = Ground Truth (SEG).
    - flair: (D,H,W) float
    - pred : (3,D,H,W) uint8/bool  -> [ET,TC,WT]
    - gt   : (3,D,H,W) uint8/bool  -> [ET,TC,WT]
    - z    : index slice (opsional); jika None dipilih otomatis (prioritas GT)
    """
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch

    # --- validasi bentuk ---
    assert pred.ndim == 4 and pred.shape[0] == 3, "pred harus (3,D,H,W)"
    assert gt.ndim   == 4 and gt.shape[0]   == 3, "gt harus (3,D,H,W)"
    D, H, W = flair.shape
    assert pred.shape[1:] == (D, H, W), "pred & flair harus ukuran sama"
    assert gt.shape[1:]   == (D, H, W), "gt & flair harus ukuran sama"

    # --- pilih slice otomatis bila perlu ---
    def pick_best_slice(_pred, _gt, prefer="gt"):
        cand = None
        if prefer == "gt" and _gt is not None and _gt.any():
            sc = _gt.any(axis=0).sum(axis=(1,2))  # (D,)
            if sc.max() > 0: return int(sc.argmax())
        if _pred is not None and _pred.any():
            sc = _pred.any(axis=0).sum(axis=(1,2))
            if sc.max() > 0: return int(sc.argmax())
        return D // 2

    if z is None:
        z = pick_best_slice(pred, gt, prefer=prefer)

    # --- windowing biar kontras enak ---
    lo, hi = np.percentile(flair, 1), np.percentile(flair, 99)

    ETp, TCp, WTp = pred[0], pred[1], pred[2]
    ETg, TCg, WTg = gt[0],   gt[1],   gt[2]

    fig, (axp, axg) = plt.subplots(1, 2, figsize=(12, 5))

    # ===== KIRI: PREDIKSI =====
    axp.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
    axp.imshow(np.ma.masked_where(ETp[z]==0, ETp[z]), cmap="Reds",   alpha=fill_alpha, vmin=0, vmax=1)
    axp.imshow(np.ma.masked_where(TCp[z]==0, TCp[z]), cmap="Greens", alpha=fill_alpha, vmin=0, vmax=1)
    axp.imshow(np.ma.masked_where(WTp[z]==0, WTp[z]), cmap="Blues",  alpha=fill_alpha, vmin=0, vmax=1)
    if ETp[z].any(): axp.contour(ETp[z], levels=[0.5], colors="#ff2d2d", linewidths=2.0)
    if TCp[z].any(): axp.contour(TCp[z], levels=[0.5], colors="#2dcc2d", linewidths=2.0)
    if WTp[z].any(): axp.contour(WTp[z], levels=[0.5], colors="#2d4dff", linewidths=2.0)
    axp.set_title(f"z={z} — Prediction")
    axp.axis("off")

    # ===== KANAN: GROUND TRUTH =====
    axg.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
    axg.imshow(np.ma.masked_where(ETg[z]==0, ETg[z]), cmap="Reds",   alpha=fill_alpha, vmin=0, vmax=1)
    axg.imshow(np.ma.masked_where(TCg[z]==0, TCg[z]), cmap="Greens", alpha=fill_alpha, vmin=0, vmax=1)
    axg.imshow(np.ma.masked_where(WTg[z]==0, WTg[z]), cmap="Blues",  alpha=fill_alpha, vmin=0, vmax=1)
    if ETg[z].any(): axg.contour(ETg[z], levels=[0.5], colors="#ff2d2d", linewidths=2.0, linestyles="--")
    if TCg[z].any(): axg.contour(TCg[z], levels=[0.5], colors="#2dcc2d", linewidths=2.0, linestyles="--")
    if WTg[z].any(): axg.contour(WTg[z], levels=[0.5], colors="#2d4dff", linewidths=2.0, linestyles="--")
    axg.set_title(f"z={z} — Ground Truth")
    axg.axis("off")

    # legend
    handles = [Patch(color="#ff2d2d", label="ET"),
               Patch(color="#2dcc2d", label="TC"),
               Patch(color="#2d4dff", label="WT")]
    if title: fig.suptitle(title, fontsize=14)
    fig.legend(handles=handles, loc="upper right")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    return z

# Pred-only (sudah ada)
show_overlay_grid(zs, cols=3, title=f"{pid} — {MODEL_TYPE.upper()} (DS Ensemble)")

# Siapkan GT dari sample_t (sudah ter-transform & align dengan img)
gt_t = sample_t["label_ettcwt"]
gt = gt_t.cpu().numpy().astype(np.uint8) if torch.is_tensor(gt_t) else gt_t.astype(np.uint8)

# Pilih slice GT (atau pakai zs supaya slice-nya sama persis dengan pred)
gt_union = gt.any(axis=0)
z_scores_gt = gt_union.sum(axis=(1,2))
zs_gt = np.linspace(0, gt.shape[1]-1, 6, dtype=int) if z_scores_gt.max()==0 else np.sort(np.argsort(-z_scores_gt)[:6])

# GT-only overlay
show_gt_overlay(zs_gt, flair=flair, gt=gt, cols=3, title=f"{pid} — Ground Truth (SEG)")

gt_t = sample_t["label_ettcwt"]
gt   = gt_t.cpu().numpy().astype(np.uint8) if torch.is_tensor(gt_t) else gt_t.astype(np.uint8)

_ = show_pred_vs_gt_pair(
    flair=flair,
    pred=pred,
    gt=gt,
    z=None,  # biarkan auto-pick slice paling informatif
    title=f"{pid} — {MODEL_TYPE.upper()} (Pred vs SEG)",
    save_path=None  # isi path kalau mau disimpan
)

gt_t = sample_t["label_ettcwt"]
gt = gt_t.cpu().numpy().astype(np.uint8) if torch.is_tensor(gt_t) else gt_t.astype(np.uint8)

# Tampilkan satu irisan, overlay Pred vs GT di satu gambar
_ = show_pred_gt_single(flair, pred, gt=gt, z=None, mode="overlay",
                        title=f"{pid} — {MODEL_TYPE.upper()} (1 slice)")


# Prediksi E5

In [ ]:
# ==== 1) Setup: pilih model & checkpoint ====
import os, math, torch, numpy as np, nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path
from monai.inferers import sliding_window_inference
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_TYPE   = "unetpp"   # "unetpp" atau "unet"
CKPT_PATH = configured_path("BTS_UNETPP_RAW_CHECKPOINT", CHECKPOINT_ROOT / "unetpp_raw" / "unetpp_medicalnet_multilabel_best_20251015-001406.pt")
THRESH       = 0.5        # sigmoid threshold untuk ET/TC/WT
ROI_DEFAULT  = (128,128,128)  # fallback roi (akan di-cap by volume size)

if MODEL_TYPE == "unetpp":
    model = UNetPP3D_MedicalNet(
        in_channels=4, out_channels=3,
        features=(16,32,64,128,128,16),
        enc_channels=(64,64,128,256),
    ).to(device)
elif MODEL_TYPE == "unet":
    # DS ON agar cocok dengan checkpoint training-mu
    model = UNet3D_MedicalNet(
        in_channels=4, out_channels=3,
        features=(16,32,64,128,128),
        enc_channels=(64,64,128,256),
        deep_supervision=True,   # <--- ON (penting)
    ).to(device)
else:
    raise ValueError("MODEL_TYPE harus 'unetpp' atau 'unet'.")

state = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(state, strict=True)
model.eval()
for m in model.modules():
    if isinstance(m, (torch.nn.BatchNorm3d, torch.nn.SyncBatchNorm)):
        m.eval()

# ==== 6) Predictor ENSAMBLE (Opsi B) ====
# Bekerja untuk:
# - UNet biasa DS ON → out = [y_full, y_/2, y_/4, y_/8] → upsample & rata-rata
# - UNet++ eval → out tensor tunggal → langsung return
def predictor_ensemble(x, weights=(1.0, 0.5, 0.25, 0.125)):
    outs = model(x)
    if not isinstance(outs, (list, tuple)):
        return outs  # mis. UNet++ eval mengembalikan single head
    # asumsi urutan: [y_full(/1), y_ds2(/2), y_ds3(/4), y_ds4(/8)]
    y0, y2, y3, y4 = outs
    tgt = y0.shape[2:]  # (D,H,W)
    y2u = F.interpolate(y2, size=tgt, mode="trilinear", align_corners=False)
    y3u = F.interpolate(y3, size=tgt, mode="trilinear", align_corners=False)
    y4u = F.interpolate(y4, size=tgt, mode="trilinear", align_corners=False)
    w = torch.tensor(weights, dtype=y0.dtype, device=y0.device)
    if w.numel() != 4:
        raise ValueError("weights harus berisi 4 angka untuk [y0, y/2, y/4, y/8].")
    w = w / w.sum()
    return w[0]*y0 + w[1]*y2u + w[2]*y3u + w[3]*y4u

# ==== 7) Ambil satu kasus & apply transforms ====
idx = 0
sample_raw = all_paths[idx]
sample_t   = test_transforms(sample_raw)
img = sample_t["image"].unsqueeze(0).to(device)  # [1,4,D,H,W]
pid = sample_t["pid"]

# ==== 8) Sliding-window inference (pakai predictor_ensemble) ====
_, _, D, H, W = img.shape
roi = (min(ROI_DEFAULT[0], D), min(ROI_DEFAULT[1], H), min(ROI_DEFAULT[2], W))
with torch.no_grad():
    logits = sliding_window_inference(
        inputs=img, roi_size=roi, sw_batch_size=1,
        predictor=predictor_ensemble, overlap=0.5, mode="gaussian"
    )  # [1,3,D,H,W]

probs = torch.sigmoid(logits)[0].cpu().numpy()  # (3,D,H,W)
pred  = (probs > THRESH).astype(np.uint8)

# --- sanity check ---
print("prob stats: min/max/mean =", float(probs.min()), float(probs.max()), float(probs.mean()))
voxel_counts = pred.reshape(3, -1).sum(1)
print("voxel counts  ET/TC/WT  =", voxel_counts.tolist())

# per-slice (axial) jumlah voxel bertopeng
per_slice = pred.any(axis=0).sum(axis=(1,2))
print("top slices by prediction:", np.argsort(-per_slice)[:10])


# ==== 9) Pilih irisan informatif ====
tumor_map = pred.any(axis=0)            # (D,H,W)
z_scores  = tumor_map.sum(axis=(1,2))   # (D,)
if z_scores.max() == 0:
    zs = np.linspace(0, D-1, 6, dtype=int)
else:
    zs = np.sort(np.argsort(-z_scores)[:6])

# ==== 10) Visualisasi overlay (FLAIR) ====
flair = img[0,0].cpu().numpy()
ET, TC, WT = pred[0], pred[1], pred[2]



def pick_best_slice(flair, pred=None, gt=None, prefer="gt"):
    """
    Pilih irisan (z) yang paling 'ramai' tumor.
    Urutan preferensi: GT -> Pred -> tengah volume.
    """
    z = None
    if prefer == "gt" and gt is not None:
        uni = gt.any(axis=0)                      # (D,H,W)
        sc  = uni.sum(axis=(1,2))                 # (D,)
        if sc.max() > 0:
            z = int(sc.argmax())
    if z is None and pred is not None:
        uni = pred.any(axis=0)
        sc  = uni.sum(axis=(1,2))
        if sc.max() > 0:
            z = int(sc.argmax())
    if z is None:
        z = flair.shape[0] // 2
    return z

def show_pred_gt_single(flair, pred, gt=None, z=None, mode="overlay", title=None, save_path=None):
    """
    mode='overlay'  -> 1 gambar, Pred (solid) + GT (putus-putus) di axis yang sama.
    mode='split'    -> 2 gambar berdampingan: kiri=Pred, kanan=GT.
    """
    # Siapkan slice index
    if z is None:
        z = pick_best_slice(flair, pred=pred, gt=gt, prefer="gt")  # pilih otomatis

    lo, hi = np.percentile(flair, 1), np.percentile(flair, 99)
    ETp, TCp, WTp = pred[0], pred[1], pred[2]
    ETg = TCg = WTg = None
    if gt is not None:
        ETg, TCg, WTg = gt[0], gt[1], gt[2]

    if mode == "split":
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axp, axg = axes

        # ---- PRED (kiri)
        axp.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        axp.imshow(np.ma.masked_where(ETp[z]==0, ETp[z]), cmap="Reds",   alpha=0.25, vmin=0, vmax=1)
        axp.imshow(np.ma.masked_where(TCp[z]==0, TCp[z]), cmap="Greens", alpha=0.25, vmin=0, vmax=1)
        axp.imshow(np.ma.masked_where(WTp[z]==0, WTp[z]), cmap="Blues",  alpha=0.25, vmin=0, vmax=1)
        if ETp[z].any(): axp.contour(ETp[z], levels=[0.5], colors="#ff2d2d", linewidths=2.0)
        if TCp[z].any(): axp.contour(TCp[z], levels=[0.5], colors="#2dcc2d", linewidths=2.0)
        if WTp[z].any(): axp.contour(WTp[z], levels=[0.5], colors="#2d4dff", linewidths=2.0)
        axp.set_title(f"z={z} — Prediction")
        axp.axis("off")

        # ---- GT (kanan)
        axg.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        if gt is not None:
            axg.imshow(np.ma.masked_where(ETg[z]==0, ETg[z]), cmap="Reds",   alpha=0.25, vmin=0, vmax=1)
            axg.imshow(np.ma.masked_where(TCg[z]==0, TCg[z]), cmap="Greens", alpha=0.25, vmin=0, vmax=1)
            axg.imshow(np.ma.masked_where(WTg[z]==0, WTg[z]), cmap="Blues",  alpha=0.25, vmin=0, vmax=1)
            if ETg[z].any(): axg.contour(ETg[z], levels=[0.5], colors="#ff2d2d", linewidths=2, linestyles="--")
            if TCg[z].any(): axg.contour(TCg[z], levels=[0.5], colors="#2dcc2d", linewidths=2, linestyles="--")
            if WTg[z].any(): axg.contour(WTg[z], levels=[0.5], colors="#2d4dff", linewidths=2, linestyles="--")
        axg.set_title(f"z={z} — Ground Truth")
        axg.axis("off")

        if title: fig.suptitle(title, fontsize=14)
        plt.tight_layout()
        if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()

    else:  # mode == "overlay"
        fig, ax = plt.subplots(1, 1, figsize=(6, 6))
        ax.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        # Pred (solid)
        if ETp[z].any(): ax.contour(ETp[z], levels=[0.5], colors="#ff2d2d", linewidths=2.0)
        if TCp[z].any(): ax.contour(TCp[z], levels=[0.5], colors="#2dcc2d", linewidths=2.0)
        if WTp[z].any(): ax.contour(WTp[z], levels=[0.5], colors="#2d4dff", linewidths=2.0)
        # GT (dashed)
        if gt is not None:
            if ETg[z].any(): ax.contour(ETg[z], levels=[0.5], colors="#ff2d2d", linewidths=2.0, linestyles="--")
            if TCg[z].any(): ax.contour(TCg[z], levels=[0.5], colors="#2dcc2d", linewidths=2.0, linestyles="--")
            if WTg[z].any(): ax.contour(WTg[z], levels=[0.5], colors="#2d4dff", linewidths=2.0, linestyles="--")

        ax.set_title(f"z={z} — Pred (solid) vs GT (dashed)")
        ax.axis("off")
        if title: fig.suptitle(title, fontsize=14)
        plt.tight_layout()
        if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()

    print(f"[INFO] Menampilkan slice z={z}")
    return z

def show_overlay_grid(z_indices, cols=3, title=None, save_path=None):
    rows = math.ceil(len(z_indices)/cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
    axes = np.array(axes).reshape(rows, cols)

    # normalisasi window supaya kontras
    lo, hi = np.percentile(flair, 1), np.percentile(flair, 99)

    for i, z in enumerate(z_indices):
        r, c = divmod(i, cols)
        ax = axes[r, c]
        ax.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)

        # kontur tebal
        if ET[z].any():
            ax.contour(ET[z], levels=[0.5], colors='#ff2d2d', linewidths=2.0)
        if TC[z].any():
            ax.contour(TC[z], levels=[0.5], colors='#2dcc2d', linewidths=2.0)
        if WT[z].any():
            ax.contour(WT[z], levels=[0.5], colors='#2d4dff', linewidths=2.0)

        # overlay fill tipis (opsional)
        et_fill = np.ma.masked_where(ET[z]==0, ET[z])
        tc_fill = np.ma.masked_where(TC[z]==0, TC[z])
        wt_fill = np.ma.masked_where(WT[z]==0, WT[z])
        ax.imshow(et_fill, cmap='Reds',   alpha=0.25, vmin=0, vmax=1)
        ax.imshow(tc_fill, cmap='Greens', alpha=0.25, vmin=0, vmax=1)
        ax.imshow(wt_fill, cmap='Blues',  alpha=0.25, vmin=0, vmax=1)

        ax.set_title(f"z={z} | ET red, TC green, WT blue")
        ax.axis("off")

    from matplotlib.patches import Patch
    handles = [Patch(color='#ff2d2d', label='ET'),
               Patch(color='#2dcc2d', label='TC'),
               Patch(color='#2d4dff', label='WT')]
    if title: fig.suptitle(title, fontsize=14)
    fig.legend(handles=handles, loc='upper right')
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

def show_gt_overlay(z_indices, flair, gt, cols=3, title=None, save_path=None):
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch
    import math

    ETg, TCg, WTg = gt[0], gt[1], gt[2]
    rows = math.ceil(len(z_indices)/cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
    axes = np.array(axes).reshape(rows, cols)
    lo, hi = np.percentile(flair, 1), np.percentile(flair, 99)

    for i, z in enumerate(z_indices):
        r, c = divmod(i, cols)
        ax = axes[r, c]
        ax.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        # fill tipis
        ax.imshow(np.ma.masked_where(ETg[z]==0, ETg[z]), cmap='Reds',   alpha=0.25, vmin=0, vmax=1)
        ax.imshow(np.ma.masked_where(TCg[z]==0, TCg[z]), cmap='Greens', alpha=0.25, vmin=0, vmax=1)
        ax.imshow(np.ma.masked_where(WTg[z]==0, WTg[z]), cmap='Blues',  alpha=0.25, vmin=0, vmax=1)
        # kontur putus-putus
        if ETg[z].any(): ax.contour(ETg[z], levels=[0.5], colors='#ff2d2d', linewidths=2, linestyles='--')
        if TCg[z].any(): ax.contour(TCg[z], levels=[0.5], colors='#2dcc2d', linewidths=2, linestyles='--')
        if WTg[z].any(): ax.contour(WTg[z], levels=[0.5], colors='#2d4dff', linewidths=2, linestyles='--')
        ax.set_title(f"z={z} | GT")
        ax.axis("off")

    handles = [Patch(color='#ff2d2d', label='ET (GT)'),
               Patch(color='#2dcc2d', label='TC (GT)'),
               Patch(color='#2d4dff', label='WT (GT)')]
    if title: fig.suptitle(title, fontsize=14)
    fig.legend(handles=handles, loc='upper right')
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


def show_pred_vs_gt(z_indices, flair, pred, gt, title=None, save_path=None):
    """Grid 2 kolom per slice: kiri = Pred, kanan = GT."""
    import numpy as np
    import matplotlib.pyplot as plt
    import math

    ETp, TCp, WTp = pred[0], pred[1], pred[2]
    ETg, TCg, WTg = gt[0],   gt[1],   gt[2]

    rows, cols = len(z_indices), 2
    fig, axes = plt.subplots(rows, cols, figsize=(8*cols, 4*rows))
    lo, hi = np.percentile(flair, 1), np.percentile(flair, 99)

    for i, z in enumerate(z_indices):
        # ---- Pred (kiri)
        axp = axes[i, 0]
        axp.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        axp.imshow(np.ma.masked_where(ETp[z]==0, ETp[z]), cmap='Reds',   alpha=0.25, vmin=0, vmax=1)
        axp.imshow(np.ma.masked_where(TCp[z]==0, TCp[z]), cmap='Greens', alpha=0.25, vmin=0, vmax=1)
        axp.imshow(np.ma.masked_where(WTp[z]==0, WTp[z]), cmap='Blues',  alpha=0.25, vmin=0, vmax=1)
        if ETp[z].any(): axp.contour(ETp[z], levels=[0.5], colors='#ff2d2d', linewidths=2.0)
        if TCp[z].any(): axp.contour(TCp[z], levels=[0.5], colors='#2dcc2d', linewidths=2.0)
        if WTp[z].any(): axp.contour(WTp[z], levels=[0.5], colors='#2d4dff', linewidths=2.0)
        axp.set_title(f"z={z} — Pred")
        axp.axis("off")

        # ---- GT (kanan)
        axg = axes[i, 1]
        axg.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        axg.imshow(np.ma.masked_where(ETg[z]==0, ETg[z]), cmap='Reds',   alpha=0.25, vmin=0, vmax=1)
        axg.imshow(np.ma.masked_where(TCg[z]==0, TCg[z]), cmap='Greens', alpha=0.25, vmin=0, vmax=1)
        axg.imshow(np.ma.masked_where(WTg[z]==0, WTg[z]), cmap='Blues',  alpha=0.25, vmin=0, vmax=1)
        if ETg[z].any(): axg.contour(ETg[z], levels=[0.5], colors='#ff2d2d', linewidths=2, linestyles='--')
        if TCg[z].any(): axg.contour(TCg[z], levels=[0.5], colors='#2dcc2d', linewidths=2, linestyles='--')
        if WTg[z].any(): axg.contour(WTg[z], levels=[0.5], colors='#2d4dff', linewidths=2, linestyles='--')
        axg.set_title(f"z={z} — GT")
        axg.axis("off")

    if title: fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    
def show_pred_vs_gt_pair(flair, pred, gt, z=None, title=None, save_path=None,
                         fill_alpha=0.25, prefer="gt"):
    """
    Tampilkan 1 slice berdampingan: kiri = Prediksi, kanan = Ground Truth (SEG).
    - flair: (D,H,W) float
    - pred : (3,D,H,W) uint8/bool  -> [ET,TC,WT]
    - gt   : (3,D,H,W) uint8/bool  -> [ET,TC,WT]
    - z    : index slice (opsional); jika None dipilih otomatis (prioritas GT)
    """
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch

    # --- validasi bentuk ---
    assert pred.ndim == 4 and pred.shape[0] == 3, "pred harus (3,D,H,W)"
    assert gt.ndim   == 4 and gt.shape[0]   == 3, "gt harus (3,D,H,W)"
    D, H, W = flair.shape
    assert pred.shape[1:] == (D, H, W), "pred & flair harus ukuran sama"
    assert gt.shape[1:]   == (D, H, W), "gt & flair harus ukuran sama"

    # --- pilih slice otomatis bila perlu ---
    def pick_best_slice(_pred, _gt, prefer="gt"):
        cand = None
        if prefer == "gt" and _gt is not None and _gt.any():
            sc = _gt.any(axis=0).sum(axis=(1,2))  # (D,)
            if sc.max() > 0: return int(sc.argmax())
        if _pred is not None and _pred.any():
            sc = _pred.any(axis=0).sum(axis=(1,2))
            if sc.max() > 0: return int(sc.argmax())
        return D // 2

    if z is None:
        z = pick_best_slice(pred, gt, prefer=prefer)

    # --- windowing biar kontras enak ---
    lo, hi = np.percentile(flair, 1), np.percentile(flair, 99)

    ETp, TCp, WTp = pred[0], pred[1], pred[2]
    ETg, TCg, WTg = gt[0],   gt[1],   gt[2]

    fig, (axp, axg) = plt.subplots(1, 2, figsize=(12, 5))

    # ===== KIRI: PREDIKSI =====
    axp.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
    axp.imshow(np.ma.masked_where(ETp[z]==0, ETp[z]), cmap="Reds",   alpha=fill_alpha, vmin=0, vmax=1)
    axp.imshow(np.ma.masked_where(TCp[z]==0, TCp[z]), cmap="Greens", alpha=fill_alpha, vmin=0, vmax=1)
    axp.imshow(np.ma.masked_where(WTp[z]==0, WTp[z]), cmap="Blues",  alpha=fill_alpha, vmin=0, vmax=1)
    if ETp[z].any(): axp.contour(ETp[z], levels=[0.5], colors="#ff2d2d", linewidths=2.0)
    if TCp[z].any(): axp.contour(TCp[z], levels=[0.5], colors="#2dcc2d", linewidths=2.0)
    if WTp[z].any(): axp.contour(WTp[z], levels=[0.5], colors="#2d4dff", linewidths=2.0)
    axp.set_title(f"z={z} — Prediction")
    axp.axis("off")

    # ===== KANAN: GROUND TRUTH =====
    axg.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
    axg.imshow(np.ma.masked_where(ETg[z]==0, ETg[z]), cmap="Reds",   alpha=fill_alpha, vmin=0, vmax=1)
    axg.imshow(np.ma.masked_where(TCg[z]==0, TCg[z]), cmap="Greens", alpha=fill_alpha, vmin=0, vmax=1)
    axg.imshow(np.ma.masked_where(WTg[z]==0, WTg[z]), cmap="Blues",  alpha=fill_alpha, vmin=0, vmax=1)
    if ETg[z].any(): axg.contour(ETg[z], levels=[0.5], colors="#ff2d2d", linewidths=2.0, linestyles="--")
    if TCg[z].any(): axg.contour(TCg[z], levels=[0.5], colors="#2dcc2d", linewidths=2.0, linestyles="--")
    if WTg[z].any(): axg.contour(WTg[z], levels=[0.5], colors="#2d4dff", linewidths=2.0, linestyles="--")
    axg.set_title(f"z={z} — Ground Truth")
    axg.axis("off")

    # legend
    handles = [Patch(color="#ff2d2d", label="ET"),
               Patch(color="#2dcc2d", label="TC"),
               Patch(color="#2d4dff", label="WT")]
    if title: fig.suptitle(title, fontsize=14)
    fig.legend(handles=handles, loc="upper right")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    return z

# Pred-only (sudah ada)
show_overlay_grid(zs, cols=3, title=f"{pid} — {MODEL_TYPE.upper()} (DS Ensemble)")

# Siapkan GT dari sample_t (sudah ter-transform & align dengan img)
gt_t = sample_t["label_ettcwt"]
gt = gt_t.cpu().numpy().astype(np.uint8) if torch.is_tensor(gt_t) else gt_t.astype(np.uint8)

# Pilih slice GT (atau pakai zs supaya slice-nya sama persis dengan pred)
gt_union = gt.any(axis=0)
z_scores_gt = gt_union.sum(axis=(1,2))
zs_gt = np.linspace(0, gt.shape[1]-1, 6, dtype=int) if z_scores_gt.max()==0 else np.sort(np.argsort(-z_scores_gt)[:6])

# GT-only overlay
show_gt_overlay(zs_gt, flair=flair, gt=gt, cols=3, title=f"{pid} — Ground Truth (SEG)")

gt_t = sample_t["label_ettcwt"]
gt   = gt_t.cpu().numpy().astype(np.uint8) if torch.is_tensor(gt_t) else gt_t.astype(np.uint8)

_ = show_pred_vs_gt_pair(
    flair=flair,
    pred=pred,
    gt=gt,
    z=None,  # biarkan auto-pick slice paling informatif
    title=f"{pid} — {MODEL_TYPE.upper()} (Pred vs SEG)",
    save_path=None  # isi path kalau mau disimpan
)

gt_t = sample_t["label_ettcwt"]
gt = gt_t.cpu().numpy().astype(np.uint8) if torch.is_tensor(gt_t) else gt_t.astype(np.uint8)

# Tampilkan satu irisan, overlay Pred vs GT di satu gambar
_ = show_pred_gt_single(flair, pred, gt=gt, z=None, mode="overlay",
                        title=f"{pid} — {MODEL_TYPE.upper()} (1 slice)")


# Prediksi E6

In [ ]:
# ==== 1) Setup: pilih model & checkpoint ====
import os, math, torch, numpy as np, nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path
from monai.inferers import sliding_window_inference
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_TYPE   = "unet"   # "unetpp" atau "unet"
CKPT_PATH = configured_path("BTS_UNET_RAW_CHECKPOINT", CHECKPOINT_ROOT / "unet_raw" / "unetpp_medicalnet_multilabel_best_20251015-041427.pt")
THRESH       = 0.5        # sigmoid threshold untuk ET/TC/WT
ROI_DEFAULT  = (128,128,128)  # fallback roi (akan di-cap by volume size)

if MODEL_TYPE == "unetpp":
    model = UNetPP3D_MedicalNet(
        in_channels=4, out_channels=3,
        features=(16,32,64,128,128,16),
        enc_channels=(64,64,128,256),
    ).to(device)
elif MODEL_TYPE == "unet":
    # DS ON agar cocok dengan checkpoint training-mu
    model = UNet3D_MedicalNet(
        in_channels=4, out_channels=3,
        features=(16,32,64,128,128),
        enc_channels=(64,64,128,256),
        deep_supervision=True,   # <--- ON (penting)
    ).to(device)
else:
    raise ValueError("MODEL_TYPE harus 'unetpp' atau 'unet'.")

state = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(state, strict=True)
model.eval()
for m in model.modules():
    if isinstance(m, (torch.nn.BatchNorm3d, torch.nn.SyncBatchNorm)):
        m.eval()

# ==== 6) Predictor ENSAMBLE (Opsi B) ====
# Bekerja untuk:
# - UNet biasa DS ON → out = [y_full, y_/2, y_/4, y_/8] → upsample & rata-rata
# - UNet++ eval → out tensor tunggal → langsung return
def predictor_ensemble(x, weights=(1.0, 0.5, 0.25, 0.125)):
    outs = model(x)
    if not isinstance(outs, (list, tuple)):
        return outs  # mis. UNet++ eval mengembalikan single head
    # asumsi urutan: [y_full(/1), y_ds2(/2), y_ds3(/4), y_ds4(/8)]
    y0, y2, y3, y4 = outs
    tgt = y0.shape[2:]  # (D,H,W)
    y2u = F.interpolate(y2, size=tgt, mode="trilinear", align_corners=False)
    y3u = F.interpolate(y3, size=tgt, mode="trilinear", align_corners=False)
    y4u = F.interpolate(y4, size=tgt, mode="trilinear", align_corners=False)
    w = torch.tensor(weights, dtype=y0.dtype, device=y0.device)
    if w.numel() != 4:
        raise ValueError("weights harus berisi 4 angka untuk [y0, y/2, y/4, y/8].")
    w = w / w.sum()
    return w[0]*y0 + w[1]*y2u + w[2]*y3u + w[3]*y4u

# ==== 7) Ambil satu kasus & apply transforms ====
idx = 0
sample_raw = all_paths[idx]
sample_t   = test_transforms(sample_raw)
img = sample_t["image"].unsqueeze(0).to(device)  # [1,4,D,H,W]
pid = sample_t["pid"]

# ==== 8) Sliding-window inference (pakai predictor_ensemble) ====
_, _, D, H, W = img.shape
roi = (min(ROI_DEFAULT[0], D), min(ROI_DEFAULT[1], H), min(ROI_DEFAULT[2], W))
with torch.no_grad():
    logits = sliding_window_inference(
        inputs=img, roi_size=roi, sw_batch_size=1,
        predictor=predictor_ensemble, overlap=0.5, mode="gaussian"
    )  # [1,3,D,H,W]

probs = torch.sigmoid(logits)[0].cpu().numpy()  # (3,D,H,W)
pred  = (probs > THRESH).astype(np.uint8)

# --- sanity check ---
print("prob stats: min/max/mean =", float(probs.min()), float(probs.max()), float(probs.mean()))
voxel_counts = pred.reshape(3, -1).sum(1)
print("voxel counts  ET/TC/WT  =", voxel_counts.tolist())

# per-slice (axial) jumlah voxel bertopeng
per_slice = pred.any(axis=0).sum(axis=(1,2))
print("top slices by prediction:", np.argsort(-per_slice)[:10])


# ==== 9) Pilih irisan informatif ====
tumor_map = pred.any(axis=0)            # (D,H,W)
z_scores  = tumor_map.sum(axis=(1,2))   # (D,)
if z_scores.max() == 0:
    zs = np.linspace(0, D-1, 6, dtype=int)
else:
    zs = np.sort(np.argsort(-z_scores)[:6])

# ==== 10) Visualisasi overlay (FLAIR) ====
flair = img[0,0].cpu().numpy()
ET, TC, WT = pred[0], pred[1], pred[2]



def pick_best_slice(flair, pred=None, gt=None, prefer="gt"):
    """
    Pilih irisan (z) yang paling 'ramai' tumor.
    Urutan preferensi: GT -> Pred -> tengah volume.
    """
    z = None
    if prefer == "gt" and gt is not None:
        uni = gt.any(axis=0)                      # (D,H,W)
        sc  = uni.sum(axis=(1,2))                 # (D,)
        if sc.max() > 0:
            z = int(sc.argmax())
    if z is None and pred is not None:
        uni = pred.any(axis=0)
        sc  = uni.sum(axis=(1,2))
        if sc.max() > 0:
            z = int(sc.argmax())
    if z is None:
        z = flair.shape[0] // 2
    return z

def show_pred_gt_single(flair, pred, gt=None, z=None, mode="overlay", title=None, save_path=None):
    """
    mode='overlay'  -> 1 gambar, Pred (solid) + GT (putus-putus) di axis yang sama.
    mode='split'    -> 2 gambar berdampingan: kiri=Pred, kanan=GT.
    """
    # Siapkan slice index
    if z is None:
        z = pick_best_slice(flair, pred=pred, gt=gt, prefer="gt")  # pilih otomatis

    lo, hi = np.percentile(flair, 1), np.percentile(flair, 99)
    ETp, TCp, WTp = pred[0], pred[1], pred[2]
    ETg = TCg = WTg = None
    if gt is not None:
        ETg, TCg, WTg = gt[0], gt[1], gt[2]

    if mode == "split":
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axp, axg = axes

        # ---- PRED (kiri)
        axp.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        axp.imshow(np.ma.masked_where(ETp[z]==0, ETp[z]), cmap="Reds",   alpha=0.25, vmin=0, vmax=1)
        axp.imshow(np.ma.masked_where(TCp[z]==0, TCp[z]), cmap="Greens", alpha=0.25, vmin=0, vmax=1)
        axp.imshow(np.ma.masked_where(WTp[z]==0, WTp[z]), cmap="Blues",  alpha=0.25, vmin=0, vmax=1)
        if ETp[z].any(): axp.contour(ETp[z], levels=[0.5], colors="#ff2d2d", linewidths=2.0)
        if TCp[z].any(): axp.contour(TCp[z], levels=[0.5], colors="#2dcc2d", linewidths=2.0)
        if WTp[z].any(): axp.contour(WTp[z], levels=[0.5], colors="#2d4dff", linewidths=2.0)
        axp.set_title(f"z={z} — Prediction")
        axp.axis("off")

        # ---- GT (kanan)
        axg.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        if gt is not None:
            axg.imshow(np.ma.masked_where(ETg[z]==0, ETg[z]), cmap="Reds",   alpha=0.25, vmin=0, vmax=1)
            axg.imshow(np.ma.masked_where(TCg[z]==0, TCg[z]), cmap="Greens", alpha=0.25, vmin=0, vmax=1)
            axg.imshow(np.ma.masked_where(WTg[z]==0, WTg[z]), cmap="Blues",  alpha=0.25, vmin=0, vmax=1)
            if ETg[z].any(): axg.contour(ETg[z], levels=[0.5], colors="#ff2d2d", linewidths=2, linestyles="--")
            if TCg[z].any(): axg.contour(TCg[z], levels=[0.5], colors="#2dcc2d", linewidths=2, linestyles="--")
            if WTg[z].any(): axg.contour(WTg[z], levels=[0.5], colors="#2d4dff", linewidths=2, linestyles="--")
        axg.set_title(f"z={z} — Ground Truth")
        axg.axis("off")

        if title: fig.suptitle(title, fontsize=14)
        plt.tight_layout()
        if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()

    else:  # mode == "overlay"
        fig, ax = plt.subplots(1, 1, figsize=(6, 6))
        ax.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        # Pred (solid)
        if ETp[z].any(): ax.contour(ETp[z], levels=[0.5], colors="#ff2d2d", linewidths=2.0)
        if TCp[z].any(): ax.contour(TCp[z], levels=[0.5], colors="#2dcc2d", linewidths=2.0)
        if WTp[z].any(): ax.contour(WTp[z], levels=[0.5], colors="#2d4dff", linewidths=2.0)
        # GT (dashed)
        if gt is not None:
            if ETg[z].any(): ax.contour(ETg[z], levels=[0.5], colors="#ff2d2d", linewidths=2.0, linestyles="--")
            if TCg[z].any(): ax.contour(TCg[z], levels=[0.5], colors="#2dcc2d", linewidths=2.0, linestyles="--")
            if WTg[z].any(): ax.contour(WTg[z], levels=[0.5], colors="#2d4dff", linewidths=2.0, linestyles="--")

        ax.set_title(f"z={z} — Pred (solid) vs GT (dashed)")
        ax.axis("off")
        if title: fig.suptitle(title, fontsize=14)
        plt.tight_layout()
        if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()

    print(f"[INFO] Menampilkan slice z={z}")
    return z

def show_overlay_grid(z_indices, cols=3, title=None, save_path=None):
    rows = math.ceil(len(z_indices)/cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
    axes = np.array(axes).reshape(rows, cols)

    # normalisasi window supaya kontras
    lo, hi = np.percentile(flair, 1), np.percentile(flair, 99)

    for i, z in enumerate(z_indices):
        r, c = divmod(i, cols)
        ax = axes[r, c]
        ax.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)

        # kontur tebal
        if ET[z].any():
            ax.contour(ET[z], levels=[0.5], colors='#ff2d2d', linewidths=2.0)
        if TC[z].any():
            ax.contour(TC[z], levels=[0.5], colors='#2dcc2d', linewidths=2.0)
        if WT[z].any():
            ax.contour(WT[z], levels=[0.5], colors='#2d4dff', linewidths=2.0)

        # overlay fill tipis (opsional)
        et_fill = np.ma.masked_where(ET[z]==0, ET[z])
        tc_fill = np.ma.masked_where(TC[z]==0, TC[z])
        wt_fill = np.ma.masked_where(WT[z]==0, WT[z])
        ax.imshow(et_fill, cmap='Reds',   alpha=0.25, vmin=0, vmax=1)
        ax.imshow(tc_fill, cmap='Greens', alpha=0.25, vmin=0, vmax=1)
        ax.imshow(wt_fill, cmap='Blues',  alpha=0.25, vmin=0, vmax=1)

        ax.set_title(f"z={z} | ET red, TC green, WT blue")
        ax.axis("off")

    from matplotlib.patches import Patch
    handles = [Patch(color='#ff2d2d', label='ET'),
               Patch(color='#2dcc2d', label='TC'),
               Patch(color='#2d4dff', label='WT')]
    if title: fig.suptitle(title, fontsize=14)
    fig.legend(handles=handles, loc='upper right')
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

def show_gt_overlay(z_indices, flair, gt, cols=3, title=None, save_path=None):
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch
    import math

    ETg, TCg, WTg = gt[0], gt[1], gt[2]
    rows = math.ceil(len(z_indices)/cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
    axes = np.array(axes).reshape(rows, cols)
    lo, hi = np.percentile(flair, 1), np.percentile(flair, 99)

    for i, z in enumerate(z_indices):
        r, c = divmod(i, cols)
        ax = axes[r, c]
        ax.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        # fill tipis
        ax.imshow(np.ma.masked_where(ETg[z]==0, ETg[z]), cmap='Reds',   alpha=0.25, vmin=0, vmax=1)
        ax.imshow(np.ma.masked_where(TCg[z]==0, TCg[z]), cmap='Greens', alpha=0.25, vmin=0, vmax=1)
        ax.imshow(np.ma.masked_where(WTg[z]==0, WTg[z]), cmap='Blues',  alpha=0.25, vmin=0, vmax=1)
        # kontur putus-putus
        if ETg[z].any(): ax.contour(ETg[z], levels=[0.5], colors='#ff2d2d', linewidths=2, linestyles='--')
        if TCg[z].any(): ax.contour(TCg[z], levels=[0.5], colors='#2dcc2d', linewidths=2, linestyles='--')
        if WTg[z].any(): ax.contour(WTg[z], levels=[0.5], colors='#2d4dff', linewidths=2, linestyles='--')
        ax.set_title(f"z={z} | GT")
        ax.axis("off")

    handles = [Patch(color='#ff2d2d', label='ET (GT)'),
               Patch(color='#2dcc2d', label='TC (GT)'),
               Patch(color='#2d4dff', label='WT (GT)')]
    if title: fig.suptitle(title, fontsize=14)
    fig.legend(handles=handles, loc='upper right')
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


def show_pred_vs_gt(z_indices, flair, pred, gt, title=None, save_path=None):
    """Grid 2 kolom per slice: kiri = Pred, kanan = GT."""
    import numpy as np
    import matplotlib.pyplot as plt
    import math

    ETp, TCp, WTp = pred[0], pred[1], pred[2]
    ETg, TCg, WTg = gt[0],   gt[1],   gt[2]

    rows, cols = len(z_indices), 2
    fig, axes = plt.subplots(rows, cols, figsize=(8*cols, 4*rows))
    lo, hi = np.percentile(flair, 1), np.percentile(flair, 99)

    for i, z in enumerate(z_indices):
        # ---- Pred (kiri)
        axp = axes[i, 0]
        axp.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        axp.imshow(np.ma.masked_where(ETp[z]==0, ETp[z]), cmap='Reds',   alpha=0.25, vmin=0, vmax=1)
        axp.imshow(np.ma.masked_where(TCp[z]==0, TCp[z]), cmap='Greens', alpha=0.25, vmin=0, vmax=1)
        axp.imshow(np.ma.masked_where(WTp[z]==0, WTp[z]), cmap='Blues',  alpha=0.25, vmin=0, vmax=1)
        if ETp[z].any(): axp.contour(ETp[z], levels=[0.5], colors='#ff2d2d', linewidths=2.0)
        if TCp[z].any(): axp.contour(TCp[z], levels=[0.5], colors='#2dcc2d', linewidths=2.0)
        if WTp[z].any(): axp.contour(WTp[z], levels=[0.5], colors='#2d4dff', linewidths=2.0)
        axp.set_title(f"z={z} — Pred")
        axp.axis("off")

        # ---- GT (kanan)
        axg = axes[i, 1]
        axg.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
        axg.imshow(np.ma.masked_where(ETg[z]==0, ETg[z]), cmap='Reds',   alpha=0.25, vmin=0, vmax=1)
        axg.imshow(np.ma.masked_where(TCg[z]==0, TCg[z]), cmap='Greens', alpha=0.25, vmin=0, vmax=1)
        axg.imshow(np.ma.masked_where(WTg[z]==0, WTg[z]), cmap='Blues',  alpha=0.25, vmin=0, vmax=1)
        if ETg[z].any(): axg.contour(ETg[z], levels=[0.5], colors='#ff2d2d', linewidths=2, linestyles='--')
        if TCg[z].any(): axg.contour(TCg[z], levels=[0.5], colors='#2dcc2d', linewidths=2, linestyles='--')
        if WTg[z].any(): axg.contour(WTg[z], levels=[0.5], colors='#2d4dff', linewidths=2, linestyles='--')
        axg.set_title(f"z={z} — GT")
        axg.axis("off")

    if title: fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    
def show_pred_vs_gt_pair(flair, pred, gt, z=None, title=None, save_path=None,
                         fill_alpha=0.25, prefer="gt"):
    """
    Tampilkan 1 slice berdampingan: kiri = Prediksi, kanan = Ground Truth (SEG).
    - flair: (D,H,W) float
    - pred : (3,D,H,W) uint8/bool  -> [ET,TC,WT]
    - gt   : (3,D,H,W) uint8/bool  -> [ET,TC,WT]
    - z    : index slice (opsional); jika None dipilih otomatis (prioritas GT)
    """
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch

    # --- validasi bentuk ---
    assert pred.ndim == 4 and pred.shape[0] == 3, "pred harus (3,D,H,W)"
    assert gt.ndim   == 4 and gt.shape[0]   == 3, "gt harus (3,D,H,W)"
    D, H, W = flair.shape
    assert pred.shape[1:] == (D, H, W), "pred & flair harus ukuran sama"
    assert gt.shape[1:]   == (D, H, W), "gt & flair harus ukuran sama"

    # --- pilih slice otomatis bila perlu ---
    def pick_best_slice(_pred, _gt, prefer="gt"):
        cand = None
        if prefer == "gt" and _gt is not None and _gt.any():
            sc = _gt.any(axis=0).sum(axis=(1,2))  # (D,)
            if sc.max() > 0: return int(sc.argmax())
        if _pred is not None and _pred.any():
            sc = _pred.any(axis=0).sum(axis=(1,2))
            if sc.max() > 0: return int(sc.argmax())
        return D // 2

    if z is None:
        z = pick_best_slice(pred, gt, prefer=prefer)

    # --- windowing biar kontras enak ---
    lo, hi = np.percentile(flair, 1), np.percentile(flair, 99)

    ETp, TCp, WTp = pred[0], pred[1], pred[2]
    ETg, TCg, WTg = gt[0],   gt[1],   gt[2]

    fig, (axp, axg) = plt.subplots(1, 2, figsize=(12, 5))

    # ===== KIRI: PREDIKSI =====
    axp.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
    axp.imshow(np.ma.masked_where(ETp[z]==0, ETp[z]), cmap="Reds",   alpha=fill_alpha, vmin=0, vmax=1)
    axp.imshow(np.ma.masked_where(TCp[z]==0, TCp[z]), cmap="Greens", alpha=fill_alpha, vmin=0, vmax=1)
    axp.imshow(np.ma.masked_where(WTp[z]==0, WTp[z]), cmap="Blues",  alpha=fill_alpha, vmin=0, vmax=1)
    if ETp[z].any(): axp.contour(ETp[z], levels=[0.5], colors="#ff2d2d", linewidths=2.0)
    if TCp[z].any(): axp.contour(TCp[z], levels=[0.5], colors="#2dcc2d", linewidths=2.0)
    if WTp[z].any(): axp.contour(WTp[z], levels=[0.5], colors="#2d4dff", linewidths=2.0)
    axp.set_title(f"z={z} — Prediction")
    axp.axis("off")

    # ===== KANAN: GROUND TRUTH =====
    axg.imshow(flair[z], cmap="gray", vmin=lo, vmax=hi)
    axg.imshow(np.ma.masked_where(ETg[z]==0, ETg[z]), cmap="Reds",   alpha=fill_alpha, vmin=0, vmax=1)
    axg.imshow(np.ma.masked_where(TCg[z]==0, TCg[z]), cmap="Greens", alpha=fill_alpha, vmin=0, vmax=1)
    axg.imshow(np.ma.masked_where(WTg[z]==0, WTg[z]), cmap="Blues",  alpha=fill_alpha, vmin=0, vmax=1)
    if ETg[z].any(): axg.contour(ETg[z], levels=[0.5], colors="#ff2d2d", linewidths=2.0, linestyles="--")
    if TCg[z].any(): axg.contour(TCg[z], levels=[0.5], colors="#2dcc2d", linewidths=2.0, linestyles="--")
    if WTg[z].any(): axg.contour(WTg[z], levels=[0.5], colors="#2d4dff", linewidths=2.0, linestyles="--")
    axg.set_title(f"z={z} — Ground Truth")
    axg.axis("off")

    # legend
    handles = [Patch(color="#ff2d2d", label="ET"),
               Patch(color="#2dcc2d", label="TC"),
               Patch(color="#2d4dff", label="WT")]
    if title: fig.suptitle(title, fontsize=14)
    fig.legend(handles=handles, loc="upper right")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    return z

# Pred-only (sudah ada)
show_overlay_grid(zs, cols=3, title=f"{pid} — {MODEL_TYPE.upper()} (DS Ensemble)")

# Siapkan GT dari sample_t (sudah ter-transform & align dengan img)
gt_t = sample_t["label_ettcwt"]
gt = gt_t.cpu().numpy().astype(np.uint8) if torch.is_tensor(gt_t) else gt_t.astype(np.uint8)

# Pilih slice GT (atau pakai zs supaya slice-nya sama persis dengan pred)
gt_union = gt.any(axis=0)
z_scores_gt = gt_union.sum(axis=(1,2))
zs_gt = np.linspace(0, gt.shape[1]-1, 6, dtype=int) if z_scores_gt.max()==0 else np.sort(np.argsort(-z_scores_gt)[:6])

# GT-only overlay
show_gt_overlay(zs_gt, flair=flair, gt=gt, cols=3, title=f"{pid} — Ground Truth (SEG)")

gt_t = sample_t["label_ettcwt"]
gt   = gt_t.cpu().numpy().astype(np.uint8) if torch.is_tensor(gt_t) else gt_t.astype(np.uint8)

_ = show_pred_vs_gt_pair(
    flair=flair,
    pred=pred,
    gt=gt,
    z=None,  # biarkan auto-pick slice paling informatif
    title=f"{pid} — {MODEL_TYPE.upper()} (Pred vs SEG)",
    save_path=None  # isi path kalau mau disimpan
)

gt_t = sample_t["label_ettcwt"]
gt = gt_t.cpu().numpy().astype(np.uint8) if torch.is_tensor(gt_t) else gt_t.astype(np.uint8)

# Tampilkan satu irisan, overlay Pred vs GT di satu gambar
_ = show_pred_gt_single(flair, pred, gt=gt, z=None, mode="overlay",
                        title=f"{pid} — {MODEL_TYPE.upper()} (1 slice)")
